# Nanobody Generation Analysis
## SFT → DPO → GDPO cascade — full results

This notebook loads all generated CSVs (3 seeds × 3 temperatures × 4 models) and reproduces every figure and table used in the paper. All property computations were done via `run_properties.py`; this notebook is purely for visualization and verification.

**Models:**
- `SFT` — supervised fine-tuning baseline
- `DPO` — preference-aligned on binding quality
- `GDPO` — reward-optimized for liability reduction (main model, SFT→DPO→GDPO)
- `GDPO_SFT` — ablation: reward-optimized directly from SFT (no DPO)

**Seeds:** 42, 123, 456  
**Temperatures:** 0.7, 0.9, 1.2  
**Sequences per run:** ~6,400 (64 targets × 100 generations)

---
**Run inside Docker:**
```bash
docker run -it --rm --gpus all --ipc=host     -v $(pwd):/workspace -w /workspace -e PYTHONPATH=/workspace     -p 8888:8888     gdpo:latest /bin/bash
pip install jupyter matplotlib seaborn scipy
jupyter notebook --allow-root --ip=0.0.0.0 --port=8888 --no-browser

```

In [ ]:
import os
import warnings 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from collections import Counter
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Paths — auto-detects Docker (/workspace) vs host path ─────────────────
_CANDIDATES = [
    Path("/workspace/src/binder_design/protgpt2_dpo/analysis/csv/statistical"),
    Path("/home/radheesh_aikium_com/genseq2/src/binder_design/protgpt2_dpo/analysis/csv/statistical"),
]
BASE = next((p for p in _CANDIDATES if p.exists()), None)
if BASE is None:
    raise FileNotFoundError("Cannot find CSV directory. Tried:\n" + "\n".join(str(p) for p in _CANDIDATES))
print(f"✓ BASE = {BASE}")

MODELS   = ["SFT", "DPO", "GDPO", "GDPO_SFT"]
SEEDS    = [42, 123, 456]
TEMPS    = [0.7, 0.9, 1.2]

MODEL_COLORS = {
    "SFT":      "#4C72B0",
    "DPO":      "#DD8452",
    "GDPO":     "#55A868",
    "GDPO_SFT": "#C44E52",
}
MODEL_LABELS = {
    "SFT":      "SFT",
    "DPO":      "DPO",
    "GDPO":     "GDPO (DPO)",
    "GDPO_SFT": "GDPO (SFT)",
}

# ── Load all per-run profiled CSVs ─────────────────────────────────────────
def load_all(models=MODELS, seeds=SEEDS, temps=TEMPS):
    frames = []
    for m in models:
        for s in seeds:
            for t in temps:
                p = BASE / m / "properties" / f"{m}_seed{s}_temp{t}_profiled.csv"
                if p.exists():
                    df = pd.read_csv(p)
                    df["model"] = m
                    df["seed"]  = s
                    df["temperature"] = t
                    frames.append(df)
    return pd.concat(frames, ignore_index=True)

ALL = load_all()
print(f"Loaded {len(ALL):,} rows across {ALL['model'].nunique()} models")
print(ALL.groupby("model")[["model","seed","temperature"]].value_counts().groupby("model").count())
print("\nColumns:", list(ALL.columns[:10]), "...")
print(f"\nModels: {ALL['model'].unique()}")
print(f"Seeds:  {sorted(ALL['seed'].unique())}")
print(f"Temps:  {sorted(ALL['temperature'].unique())}")

## 2. Main Result — Emergent Thermostability (Tm)
Tm was predicted by TEMPRO (ESM-2 + Keras). No stage explicitly optimized Tm.

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

os.makedirs("figures", exist_ok=True)

# ── Filter valid sequences ────────────────────────────────────────────────
valid_all = ALL[ALL["is_valid_126"] == True].copy()

# Per-run means: one mean Tm per (model, seed, temperature)
tm_runs = (
    valid_all.groupby(["model", "seed", "temperature"])["tempro_tm"]
    .mean()
    .reset_index(name="tm_mean")
)

# Summary across all 9 runs
tm_summary = tm_runs.groupby("model")["tm_mean"].agg(["mean", "std"]).round(3)
tm_summary.columns = ["Tm Mean (°C)", "Tm Std"]

print("Tm across all seeds & temperatures (valid seqs only):\n")
print(tm_summary.loc[[m for m in MODELS if m in tm_summary.index]].to_string())

# ── Summary at a single reporting temperature ─────────────────────────────
REPORT_TEMP = 0.7

tm_seed = (
    valid_all[valid_all["temperature"] == REPORT_TEMP]
    .groupby(["model", "seed"])["tempro_tm"]
    .mean()
    .reset_index(name="tm_mean")
)

tm_seed_summary = tm_seed.groupby("model")["tm_mean"].agg(["mean", "std"]).round(3)
tm_seed_summary.columns = ["Tm Mean (°C)", "Tm Std"]

print(f"\nTm at temp={REPORT_TEMP} — mean ± std across 3 seeds:\n")
print(tm_seed_summary.loc[[m for m in MODELS if m in tm_seed_summary.index]].to_string())

# ── Common order and style ────────────────────────────────────────────────
order = [m for m in MODELS if m in tm_summary.index]
labels = [MODEL_LABELS[m] for m in order]
colors = [MODEL_COLORS[m] for m in order]

# Use a common y-axis range so error bars do not look exaggerated
all_means = [tm_summary.loc[m, "Tm Mean (°C)"] for m in order]
all_stds  = [tm_summary.loc[m, "Tm Std"] for m in order]

ymin = np.floor(min(all_means) - 2.0)
ymax = np.ceil(max(all_means) + 3.0)

def make_barplot(summary_df, title, ylabel, save_path):
    means = [summary_df.loc[m, "Tm Mean (°C)"] for m in order]
    stds  = [summary_df.loc[m, "Tm Std"] for m in order]

    fig, ax = plt.subplots(figsize=(7.2, 5.0))

    bars = ax.bar(
        labels,
        means,
        color=colors,
        alpha=0.82,
        edgecolor="white",
        linewidth=0.8,
        yerr=stds,
        capsize=4,
        error_kw=dict(
            ecolor="black",
            elinewidth=1.0,
            capthick=1.0,
        ),
    )

    # Value labels just above each bar's own error bar
    for bar, mean, std in zip(bars, means, stds):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            mean + std + 0.15,
            f"{mean:.1f}°C",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color="#1a1a1a",
        )

    
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_ylim(ymin, ymax)
    ax.tick_params(axis="x", rotation=15)
    

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

# ── Figure 1: 3 seeds at T=0.7 ────────────────────────────────────────────
make_barplot(
    tm_seed_summary,
    title="Tm across all 3 seeds (3 seeds at T=0.7)",
    ylabel="Tm (°C)",
    save_path="figures/fig_tm_3seeds_t07.png",
)

# ── Figure 2: all 9 runs ──────────────────────────────────────────────────
make_barplot(
    tm_summary,
    title="Tm across all 9 runs (3 seeds × 3 temperatures)",
    ylabel="Tm (°C)",
    save_path="figures/fig_tm_9runs.png",
)

# ── ΔTm relative to SFT at T=0.7 ──────────────────────────────────────────
sft_tm = tm_seed_summary.loc["SFT", "Tm Mean (°C)"]
for m in ["DPO", "GDPO", "GDPO_SFT"]:
    if m in tm_seed_summary.index:
        delta = tm_seed_summary.loc[m, "Tm Mean (°C)"] - sft_tm
        print(f"SFT → {m}: ΔTm = {delta:+.2f}°C")

In [ ]:

# ── Generic bar-plot helper (reuses make_barplot + order/labels/colors from cell above) ──

REPORT_TEMP = 0.7

def plot_metric(col, display_name, unit, invert=False, save_prefix=None,y_min=None,y_max=None):
    """
    Two bar-plots side by side for any scalar metric:
      (a) 3 seeds at REPORT_TEMP   — reproducibility
      (b) all 9 runs               — robustness

    invert=True  → lower raw value = better (e.g. instability index)
    """
    valid_col = valid_all.dropna(subset=[col])

    # Per-run means
    runs = (
        valid_col.groupby(["model", "seed", "temperature"])[col]
        .mean().reset_index(name="val")
    )
    summ_all = runs.groupby("model")["val"].agg(["mean", "std"]).round(4)
    summ_all.columns = ["Mean", "Std"]

    # Per-seed at REPORT_TEMP
    seed_runs = (
        valid_col[valid_col["temperature"] == REPORT_TEMP]
        .groupby(["model", "seed"])[col]
        .mean().reset_index(name="val")
    )
    summ_seed = seed_runs.groupby("model")["val"].agg(["mean", "std"]).round(4)
    summ_seed.columns = ["Mean", "Std"]

    avail = [m for m in MODELS if m in summ_seed.index]
    lbl   = [MODEL_LABELS[m] for m in avail]
    clr   = [MODEL_COLORS[m] for m in avail]

    all_means = [summ_all.loc[m, "Mean"] for m in avail if m in summ_all.index]
    auto_ymin = np.floor(min(all_means) - abs(min(all_means)) * 0.04)
    auto_ymax = np.ceil(max(all_means)  + abs(max(all_means)) * 0.06)

    ymin = y_min if y_min is not None else auto_ymin
    ymax = y_max if y_max is not None else auto_ymax

    def _draw(ax, summ, title):
        avail_m = [m for m in avail if m in summ.index]
        lbl_m   = [MODEL_LABELS[m] for m in avail_m]
        clr_m   = [MODEL_COLORS[m] for m in avail_m]
        means   = [summ.loc[m, "Mean"] for m in avail_m]
        stds    = [summ.loc[m, "Std"]  for m in avail_m]
        bars = ax.bar(lbl_m, means, color=clr_m, alpha=0.82,
                      edgecolor="white", linewidth=0.8,
                      yerr=stds, capsize=4,
                      error_kw=dict(ecolor="black", elinewidth=1.0, capthick=1.0))
        for bar, mean, std in zip(bars, means, stds):
            ax.text(bar.get_x() + bar.get_width() / 2, mean + std + (ymax - ymin) * 0.01,
                    f"{mean:.3f}" if abs(mean) < 1 else f"{mean:.1f}",
                    ha="center", va="bottom", fontsize=8.5,
                    fontweight="bold", color="#1a1a1a")
        ax.set_ylim(ymin, ymax)
        ax.set_ylabel(
            f"{display_name} ({unit})" if unit else display_name,
            fontsize=12
        )
        # ax.set_title(title, fontsize=9.5)
        ax.tick_params(axis="x", rotation=15)
        for sp in ax.spines.values():
            sp.set_linewidth(0.8)
        # arrow = "↓ better" if invert else "↑ better"
        # ax.text(0.97, 0.97, arrow, transform=ax.transAxes,
        #         ha="right", va="top", fontsize=8, color="#888888")

    prefix = save_prefix or col

    # Figure (a): 3 seeds at REPORT_TEMP
    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    _draw(ax, summ_seed, f"{display_name} — 3 seeds, T={REPORT_TEMP}  [reproducibility]")
    plt.tight_layout()
    plt.savefig(f"figures/fig_{prefix}_3seeds.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Figure (b): all 9 runs
    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    _draw(ax, summ_all, f"{display_name} — All 9 runs  [3 seeds × 3 temps]")
    plt.tight_layout()
    plt.savefig(f"figures/fig_{prefix}_9runs.png", dpi=300, bbox_inches="tight")
    plt.show()

    print(f"\n{display_name} — Δ relative to SFT (T={REPORT_TEMP}, 3 seeds):")
    sft_val = summ_seed.loc["SFT", "Mean"]
    for m in ["DPO", "GDPO", "GDPO_SFT"]:
        if m in summ_seed.index:
            d = summ_seed.loc[m, "Mean"] - sft_val
            print(f"  SFT → {m}: {d:+.4f}")

# ── Humanness ─────────────────────────────────────────────────────────────
plot_metric("sapiens_humanness", "Humanness", "", invert=False, save_prefix="humanness")


In [ ]:

# ── Instability Index ─────────────────────────────────────────────────────
plot_metric("instability_index", "Instability Index", "", invert=True, save_prefix="instability",y_max=40)


In [ ]:

# ── Solubility (NetSolP) ──────────────────────────────────────────────────
plot_metric("netsolp_solubility", "NetSolP Solubility", "", invert=False, save_prefix="solubility")


## 3. Liability Severity
Chemical liabilities: deamidation, isomerization, glycosylation, oxidation, integrin motifs.

In [ ]:

# ── Liability category breakdown — weighted severity ────────────────────
REPORT_TEMP = 0.7

# Severity weights from rewards.py (exactly matching scan_sequence_liabilities_core)
CATEGORY_WEIGHTS = {
    "Deamidation":      {"motif_NG":3,"motif_NS":3,"motif_NT":2,"motif_NN":2,
                         "motif_NH":2,"motif_NA":1,"motif_NQ":1},
    "Isomerization":    {"motif_DG":3,"motif_DS":2,"motif_DT":2,
                         "motif_DD":1,"motif_DH":1},
    # "Fragmentation":    {"motif_DP":3},
    "N-Glycosylation":  {"motif_N_glyco":2},
    "Oxidation":        {"motif_CDR_Met":1},
    # "Integrin":         {"motif_RGD":2,"motif_NGR":1,"motif_LDV":1},
}
CAT_COLORS = ["#4878CF","#6ACC65","#D65F5F","#B47CC7","#C4AD66","#77BEDB"]

valid_t = valid_all[valid_all["temperature"] == REPORT_TEMP].copy()
order   = [m for m in MODELS if m in valid_t["model"].unique()]
lbl     = [MODEL_LABELS[m] for m in order]
clr     = [MODEL_COLORS[m] for m in order]

def cat_severity(df_m, cat):
    weights = CATEGORY_WEIGHTS[cat]
    present = {c: w for c, w in weights.items() if c in df_m.columns}
    if not present:
        return 0.0
    return sum(df_m[c] * w for c, w in present.items()).mean()

# ── PLOT 1: Overall weighted severity per category (grouped bars) ─────────
cat_vals = {}
for cat in CATEGORY_WEIGHTS:
    cat_vals[cat] = {m: cat_severity(valid_t[valid_t["model"]==m], cat) for m in order}

cats      = list(CATEGORY_WEIGHTS.keys())
n_cats    = len(cats)
n_models  = len(order)
width     = 0.18
x_pos     = np.arange(n_cats)

fig, ax = plt.subplots(figsize=(11, 5))
for i, (m, color) in enumerate(zip(order, clr)):
    vals   = [cat_vals[cat][m] for cat in cats]
    offset = (i - n_models/2 + 0.5) * width
    ax.bar(x_pos + offset, vals, width=width, color=color,
           alpha=0.87, edgecolor="white", linewidth=0.5,
           label=MODEL_LABELS[m])

ax.set_xticks(x_pos)
ax.set_xticklabels(cats, rotation=15, ha="right")
ax.set_ylabel("Mean weighted severity per sequence")
ax.legend(fontsize=9.5, framealpha=0.9)
ax.axhline(0, color="black", linewidth=0.4)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig("figures/fig_liability_category.png", dpi=300, bbox_inches="tight")
plt.show()

# Summary table
print("\nWeighted severity per category:")
hdr = f"  {'Category':<18}" + "".join(f"{MODEL_LABELS[m]:>15}" for m in order)
print(hdr)
print("  " + "-"*(18+15*n_models))
for cat in cats:
    row = f"  {cat:<18}" + "".join(f"{cat_vals[cat][m]:>15.3f}" for m in order)
    print(row)
totals = f"  {'TOTAL':<18}" + "".join(
    f"{sum(cat_vals[c][m] for c in cats):>15.3f}" for m in order)
print("  " + "-"*(18+15*n_models))
print(totals)

# ── PLOT 2: Per-target breakdown for Deamidation + Isomerization ──────────
# Compute per-target severity for top categories
target_sev = {}
for cat in ["Deamidation", "Isomerization"]:
    target_sev[cat] = {}
    for m in order:
        df_m = valid_t[valid_t["model"] == m]
        target_sev[cat][m] = (
            df_m.groupby("peptide")
            .apply(lambda g: cat_severity(g, cat))
            .rename(m)
        )

# Rank targets by SFT Deamidation — pick top N most differentiated
sft_col = target_sev["Deamidation"][order[0]]
top_tgts = sft_col.sort_values(ascending=False).head(16).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
for ax_i, cat in enumerate(["Deamidation", "Isomerization"]):
    ax2   = axes[ax_i]
    n_t   = len(top_tgts)
    x2    = np.arange(n_t)
    w2    = 0.20
    for j, (m, color) in enumerate(zip(order, clr)):
        col  = target_sev[cat][m]
        vals = [col.get(t, 0.0) for t in top_tgts]
        ax2.bar(x2 + (j - n_models/2 + 0.5)*w2, vals, width=w2,
                color=color, alpha=0.83, edgecolor="white", linewidth=0.4,
                label=MODEL_LABELS[m] if ax_i==0 else "")
    ax2.set_xticks(x2)
    ax2.set_xticklabels(
        [t[:12]+"…" if len(t)>12 else t for t in top_tgts],
        rotation=45, ha="right", fontsize=7
    )
    ax2.set_title(f"{cat} — top 16 targets by SFT severity")
    ax2.set_ylabel("Mean weighted severity per sequence")
    if ax_i == 0:
        ax2.legend(fontsize=8.5, framealpha=0.9)
    ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig("figures/fig_liability_per_target.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

# ── Individual motif breakdown — weighted severity per motif ─────────────
# Shows weighted severity contribution for each motif across SFT → DPO → GDPO_SFT → GDPO
# One figure per liability category.

MOTIF_GROUPS = {
    "Deamidation": {
        "motif_NG": ("NG", 3),
        "motif_NS": ("NS", 3),
        "motif_NT": ("NT", 2),
        "motif_NN": ("NN", 2),
        "motif_NH": ("NH", 2),
        "motif_NA": ("NA", 1),
        "motif_NQ": ("NQ", 1),
    },
    "Isomerization": {
        "motif_DG": ("DG", 3),
        "motif_DS": ("DS", 2),
        "motif_DT": ("DT", 2),
        "motif_DD": ("DD", 1),
        "motif_DH": ("DH", 1),
    },
    "Other": {
        "motif_N_glyco":("N-Glyco", 2),
        "motif_CDR_Met":("CDR-Met", 1),
    },
}
GROUP_TITLE_COLORS = {
    "Deamidation":   "#4878CF",
    "Isomerization": "#D65F5F",
    "Other":         "#888888",
}

valid_t2 = valid_all[valid_all["temperature"] == REPORT_TEMP].copy()
order2   = [m for m in MODELS if m in valid_t2["model"].unique()]
clr2     = [MODEL_COLORS[m] for m in order2]
n_m      = len(order2)
width2   = 0.18

for group_name, motifs in MOTIF_GROUPS.items():
    present = {col: (lbl, w) for col, (lbl, w) in motifs.items()
               if col in valid_t2.columns}
    if not present:
        continue

    n_motifs = len(present)
    x = np.arange(n_motifs)
    motif_cols    = list(present.keys())
    motif_lbls    = [present[c][0] for c in motif_cols]
    motif_weights = [present[c][1] for c in motif_cols]

    mean_counts = {m: [valid_t2[valid_t2["model"]==m][c].mean() for c in motif_cols]
                   for m in order2}

    fig, ax = plt.subplots(figsize=(max(6, n_motifs * 1.4), 5))

    for i, (m, color) in enumerate(zip(order2, clr2)):
        weighted = [cnt * w for cnt, w in zip(mean_counts[m], motif_weights)]
        offset = (i - n_m / 2 + 0.5) * width2
        ax.bar(x + offset, weighted, width=width2,
               color=color, alpha=0.85, edgecolor="white",
               linewidth=0.5, label=MODEL_LABELS[m])

    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{lbl}\n(×{w})" for lbl, w in zip(motif_lbls, motif_weights)],
        rotation=25, ha="right", fontsize=9)
    ax.set_ylabel("Weighted severity contribution")
    
    ax.legend(fontsize=8.5, framealpha=0.9)
    ax.set_ylim(bottom=0)

    plt.tight_layout()
    plt.savefig(f"figures/fig_motifs_{group_name.lower()}.png",
                dpi=300, bbox_inches="tight")
    plt.show()

    # Print summary table
    print(f"\n{group_name} — weighted severity per sequence:")
    hdr2 = f"  {'Motif':<12}" + "".join(f"{MODEL_LABELS[m]:>14}" for m in order2)
    print(hdr2)
    print("  " + "-"*(12 + 14*n_m))
    for col, (mlbl, w) in present.items():
        row = f"  {mlbl:<12}" + "".join(
            f"{valid_t2[valid_t2['model']==m][col].mean() * w:>14.4f}" for m in order2)
        print(row)


In [ ]:

# ── NetSolP ESM1b — 640-seq run (SFT / DPO / GDPO)  ──────────────────────
# Uses the corrected ESM1b alphabet (cls_idx=0) on the original ~640-seq profiled CSVs.
# Note: GDPO_SFT not included here (no 640-seq run); use 6.5k statistical run when done.
import os

CSV_DIR = os.path.join(os.path.dirname(os.path.abspath(".")),
    "src/binder_design/protgpt2_dpo/analysis/csv")
CSV_DIR = "src/binder_design/protgpt2_dpo/analysis/csv"  # relative to /workspace

ORIG_MODELS = {
    "SFT":  ("SFT_20k_profiled.csv",                 "SFT_20k_netsolp_esm1b_raw.csv"),
    "DPO":  ("DPO_6k_profiled.csv",                  "DPO_6k_netsolp_esm1b_raw.csv"),
    "GDPO": ("GDPO_dpo_final_gated_profiled.csv",     "GDPO_dpo_final_gated_netsolp_esm1b_raw.csv"),
}

rows = []
for model, (prof_f, esm_f) in ORIG_MODELS.items():
    prof = pd.read_csv(f"{CSV_DIR}/{prof_f}")
    preds = pd.read_csv(f"{CSV_DIR}/{esm_f}")
    valid = prof["is_valid_126"] == True
    sol = preds["predicted_solubility"].values[valid]
    usa = preds["predicted_usability"].values[valid]
    rows.append({"model": model, "n": valid.sum(),
                 "sol_mean": sol.mean(), "sol_std": sol.std(),
                 "usa_mean": usa.mean(), "usa_std": usa.std()})

orig_df = pd.DataFrame(rows)
print("NetSolP ESM1b (640-seq, valid only):")
print(orig_df.to_string(index=False, float_format="{:.4f}".format))

# ── Bar charts: solubility + usability side-by-side ──────────────────────
orig_order  = ["SFT", "DPO", "GDPO"]
orig_labels = [MODEL_LABELS.get(m, m) for m in orig_order]
orig_colors = [MODEL_COLORS.get(m, "#888") for m in orig_order]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, metric, ylabel, title in [
    (axes[0], "sol", "NetSolP Solubility (ESM1b)",  "Solubility"),
    (axes[1], "usa", "NetSolP Usability (ESM1b)",   "E.coli Usability"),
]:
    means = orig_df.set_index("model").loc[orig_order, f"{metric}_mean"].values
    stds  = orig_df.set_index("model").loc[orig_order, f"{metric}_std"].values
    bars  = ax.bar(orig_labels, means, color=orig_colors, yerr=stds,
                   capsize=6, alpha=0.85, edgecolor="white", linewidth=0.8,
                   error_kw=dict(elinewidth=1.4, ecolor="#333333"))
    for bar, m_val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(stds)*0.5,
                f"{m_val:.3f}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{title}  (n≈640, ESM1b, valid seqs)")
    ymin = min(means) - 3*max(stds)
    ymax = max(means) + 4*max(stds)
    ax.set_ylim(ymin, ymax)
    ax.tick_params(axis="x", rotation=10)

plt.suptitle("NetSolP ESM1b  —  original 640-seq run  (SFT / DPO / GDPO)",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig("figures/fig_netsolp_esm1b_640.png", dpi=300, bbox_inches="tight")
plt.show()
print("\nTrend — Solubility:", " > ".join(
    orig_df.sort_values("sol_mean", ascending=False)["model"].tolist()))
print("Trend — Usability: ", " > ".join(
    orig_df.sort_values("usa_mean", ascending=False)["model"].tolist()))


## 5. Humanness (Sapiens) & Biophysical Properties

In [ ]:
bio_metrics = {
    "sapiens_humanness": "Humanness (Sapiens)",
    "instability_index": "Instability Index",
    "isoelectric_point": "Isoelectric Point (pI)",
    "charge_at_pH7":     "Charge @ pH7",
    "gravy":             "GRAVY",
    "expression_score":  "Expression Score",
}

n = len(bio_metrics)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

order = [m for m in MODELS if m in valid_all["model"].unique()]

for idx, (col, label) in enumerate(bio_metrics.items()):
    ax = axes[idx]
    if col not in valid_all.columns:
        ax.set_visible(False)
        continue
    run_means = (
        valid_all.groupby(["model","seed","temperature"])[col]
        .mean().reset_index(name="val")
    )
    means = [run_means[run_means["model"]==m]["val"].mean() for m in order]
    stds  = [run_means[run_means["model"]==m]["val"].std()  for m in order]
    bars = ax.bar([MODEL_LABELS[m] for m in order], means,
                  color=[MODEL_COLORS[m] for m in order],
                  yerr=stds, capsize=5, alpha=0.85, edgecolor="white")
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, mean + abs(mean)*0.005,
                f"{mean:.3f}", ha="center", va="bottom", fontsize=7.5)
    ax.set_title(label)
    ax.tick_params(axis="x", rotation=20, labelsize=8)

plt.suptitle("Biophysical & Immunogenicity Properties (mean ± std across 9 runs)", fontsize=12)
plt.tight_layout()
plt.savefig("figures/fig5_biophysical.png", dpi=150)
plt.show()

## 6. Ablation Study — GDPO vs GDPO_SFT
Same training objective, different initialization. Compares DPO-path GDPO vs direct SFT→GDPO.

In [ ]:
# Seed42/temp0.7 only (ablation run for GDPO_SFT)
ablation_df = valid_all[(valid_all["seed"]==42) & (valid_all["temperature"]==0.7)].copy()

ablation_metrics = {
    "tempro_tm":        "Tm (°C)",
    "instability_index":"Instability Index",
    "liability_severity":"Liability Severity",
    "sapiens_humanness":"Humanness",
    "gravy":            "GRAVY",
    "expression_score": "Expression Score",
}

ablation_order = [m for m in MODELS if m in ablation_df["model"].unique()]
results = {col: {m: ablation_df[ablation_df["model"]==m][col].mean()
                 for m in ablation_order if col in ablation_df.columns}
           for col in ablation_metrics}

# Print table
print(f"{'Metric':<25}" + "".join(f"{MODEL_LABELS[m]:>18}" for m in ablation_order))
print("-" * (25 + 18 * len(ablation_order)))
for col, label in ablation_metrics.items():
    row = f"{label:<25}"
    for m in ablation_order:
        v = results[col].get(m, float("nan"))
        row += f"{v:>18.3f}"
    print(row)

# Delta: GDPO vs GDPO_SFT
print("\n  Δ GDPO − GDPO_SFT:")
for col, label in ablation_metrics.items():
    g  = results[col].get("GDPO",  float("nan"))
    gs = results[col].get("GDPO_SFT", float("nan"))
    print(f"    {label:<25} {g-gs:>+.3f}")

# Figure: side-by-side Tm and Instability
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (col, label) in zip(axes, [("tempro_tm","Tm (°C)"), ("instability_index","Instability Index")]):
    vals = [results[col].get(m, float("nan")) for m in ablation_order]
    bars = ax.bar([MODEL_LABELS[m] for m in ablation_order], vals,
                  color=[MODEL_COLORS[m] for m in ablation_order],
                  alpha=0.85, edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + abs(v)*0.003,
                f"{v:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_title(f"Ablation: {label}")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("GDPO (DPO path) vs GDPO_SFT (ablation) — seed42, temp0.7", fontsize=11)
plt.tight_layout()
plt.savefig("figures/fig6_ablation.png", dpi=150)
plt.show()

## 7. Amino Acid Composition — Mechanistic Explanation
Why does the DPO path achieve higher Tm with equal liability? The substitution choices differ.

In [ ]:
AAs = list("ACDEFGHIKLMNPQRSTVWY")
aa_models = ["SFT", "GDPO", "GDPO_SFT"]

freqs = {}
for m in aa_models:
    seqs = ablation_df[ablation_df["model"]==m]["generated_sequence"].tolist()
    joined = "".join(seqs)
    total = len(joined)
    freqs[m] = {aa: joined.count(aa)/total*100 for aa in AAs}

delta = {aa: freqs["GDPO"][aa] - freqs["GDPO_SFT"][aa] for aa in AAs}
sorted_delta = sorted(delta.items(), key=lambda x: x[1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: delta bar (GDPO - GDPO_SFT)
ax = axes[0]
aas_sorted = [x[0] for x in sorted_delta]
deltas_sorted = [x[1] for x in sorted_delta]
colors = ["#55A868" if d > 0 else "#C44E52" for d in deltas_sorted]
ax.barh(aas_sorted, deltas_sorted, color=colors, alpha=0.85, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Δ Frequency % (GDPO − GDPO_SFT)")
ax.set_title("AA Composition Difference\n(green = GDPO uses more, red = GDPO_SFT uses more)")

# Right: grouped bar for key AAs
ax = axes[1]
key_aas = ["I", "F", "A", "Y", "S", "H", "Q", "L"]
x = np.arange(len(key_aas))
width = 0.25
for i, m in enumerate(aa_models):
    vals = [freqs[m][aa] for aa in key_aas]
    ax.bar(x + i*width, vals, width, label=MODEL_LABELS[m],
           color=MODEL_COLORS[m], alpha=0.85, edgecolor="white")
ax.set_xticks(x + width)
ax.set_xticklabels(key_aas)
ax.set_ylabel("Frequency (%)")
ax.set_title("Key AA Frequencies: Core-packing vs Flexible")
ax.legend(fontsize=8)
ax.axvline(3.5 + width/2, color="gray", linestyle="--", linewidth=0.8)
ax.text(1.5*width, ax.get_ylim()[1]*0.95, "Rigid/hydrophobic →", ha="center", fontsize=8, color="gray")
ax.text(5.5 + width/2, ax.get_ylim()[1]*0.95, "← Flexible/polar", ha="center", fontsize=8, color="gray")

plt.tight_layout()
plt.savefig("figures/fig7_aa_composition.png", dpi=150)
plt.show()

print("\nKey differences (GDPO vs GDPO_SFT):")
for aa, d in sorted(delta.items(), key=lambda x: abs(x[1]), reverse=True):
    if abs(d) > 0.1:
        direction = "GDPO uses more" if d > 0 else "GDPO_SFT uses more"
        print(f"  {aa}: {d:+.3f}%  ({direction})")

In [ ]:

# ── Cascade AA Trace: SFT → DPO → GDPO  vs  SFT → GDPO_SFT ──────────────
# Pinpoints at which training stage hydrophobic residues entered the model.

AAs = list("ACDEFGHIKLMNPQRSTVWY")

# Biophysical grouping
AA_GROUPS = {
    "Hydrophobic\n(core-packing)": list("ILFVAW"),
    "Aromatic":                    list("FYW"),
    "Polar uncharged":             list("NQST"),
    "Charged (+)":                 list("RKH"),
    "Charged (−)":                 list("DE"),
    "Small/special":               list("AGCPM"),
}
# Flat colour per group for annotation
GROUP_COLORS = {
    "Hydrophobic\n(core-packing)": "#8B4513",
    "Aromatic":                    "#9B4F9C",
    "Polar uncharged":             "#1A6B3C",
    "Charged (+)":                 "#154B8A",
    "Charged (−)":                 "#B22222",
    "Small/special":               "#5C5C5C",
}
AA_TO_GROUP = {aa: g for g, aas in AA_GROUPS.items() for aa in aas}

# ── Load seed42 / temp 0.7 for all 4 models ───────────────────────────────
SEED_C, TEMP_C = 42, 0.7

def get_seqs(model, seed=SEED_C, temp=TEMP_C):
    """Return sequences from the profiled CSV for one model/seed/temp."""
    p = BASE / model / "properties" / f"{model}_seed{seed}_temp{temp}_profiled.csv"
    if p.exists():
        return pd.read_csv(p)["generated_sequence"].dropna().astype(str).tolist()
    # fallback: use already-loaded dataframes
    if model == "GDPO_SFT":
        return ablation_df[ablation_df["model"] == model]["generated_sequence"].tolist()
    return valid_all[valid_all["model"] == model]["generated_sequence"].tolist()

def aa_freq(seqs):
    joined = "".join(seqs)
    total  = max(len(joined), 1)
    return {aa: joined.count(aa) / total * 100 for aa in AAs}

freqs_all = {m: aa_freq(get_seqs(m)) for m in ["SFT", "DPO", "GDPO", "GDPO_SFT"]}

# ── Step-by-step deltas ────────────────────────────────────────────────────
step1  = {aa: freqs_all["DPO"][aa]      - freqs_all["SFT"][aa]      for aa in AAs}  # DPO  vs SFT
step2  = {aa: freqs_all["GDPO"][aa]     - freqs_all["DPO"][aa]      for aa in AAs}  # GDPO vs DPO
cumul  = {aa: freqs_all["GDPO"][aa]     - freqs_all["SFT"][aa]      for aa in AAs}  # GDPO vs SFT (cumulative)
ablat  = {aa: freqs_all["GDPO_SFT"][aa] - freqs_all["SFT"][aa]      for aa in AAs}  # GDPO_SFT vs SFT

# ── Figure — 4-panel delta bars ────────────────────────────────────────────
panels = [
    (step1, "Step 1 — DPO vs SFT",       "#2D6FA5", "#C05C26"),
    (step2, "Step 2 — GDPO vs DPO",      "#2E7A50", "#C05C26"),
    (cumul, "Cumul. — GDPO vs SFT",      "#2E7A50", "#7A3B5E"),
    (ablat, "Ablation — GDPO_SFT vs SFT","#7A3B5E", "#C05C26"),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 7), sharey=True)
fig.patch.set_facecolor("white")

# Sort AAs by group then by cumulative GDPO delta
aa_order = sorted(
    AAs,
    key=lambda aa: (list(AA_GROUPS.keys()).index(AA_TO_GROUP.get(aa, "Small/special")),
                    -cumul[aa])
)

for ax, (delta, title, c_pos, c_neg) in zip(axes, panels):
    vals   = [delta[aa] for aa in aa_order]
    colors = [c_pos if v >= 0 else c_neg for v in vals]
    bars   = ax.barh(aa_order, vals, color=colors, alpha=0.85, edgecolor="white", height=0.7)
    ax.axvline(0, color="#333333", linewidth=0.8)
    ax.set_title(title, fontsize=10, fontweight="bold", pad=8)
    ax.set_xlabel("Δ Frequency %", fontsize=9)
    ax.tick_params(axis="y", labelsize=9)
    ax.tick_params(axis="x", labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)

    # Shade group bands
    prev_g, band_start = None, -0.5
    for i, aa in enumerate(aa_order):
        g = AA_TO_GROUP.get(aa, "Small/special")
        if g != prev_g and prev_g is not None:
            ax.axhspan(band_start, i - 0.5, color=GROUP_COLORS[prev_g], alpha=0.06, zorder=0)
            band_start = i - 0.5
        prev_g = g
    ax.axhspan(band_start, len(aa_order) - 0.5, color=GROUP_COLORS[prev_g], alpha=0.06, zorder=0)

# Group legend on rightmost panel
from matplotlib.patches import Patch
handles = [Patch(facecolor=c, alpha=0.4, label=g.replace("\n", " "))
           for g, c in GROUP_COLORS.items()]
axes[-1].legend(handles=handles, loc="lower right", fontsize=7.5,
                frameon=True, framealpha=0.9, edgecolor="#cccccc")

plt.suptitle(
    "Amino Acid Cascade Trace  (seed=42, temp=0.7)\n"
    "Pinpointing when hydrophobic enrichment entered the DPO path",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig("figures/fig_aa_cascade_trace.png", dpi=180, bbox_inches="tight", facecolor="white")
plt.show()

# ── Summary table — grouped by property ───────────────────────────────────
print("\n" + "="*95)
print(f"  {'AA':<4} {'Group':<26} {'SFT%':>7} {'DPO%':>7}  "
      f"{'Δ DPO-SFT':>10}  {'Δ GDPO-DPO':>11}  {'Δ GDPO-SFT':>11}  {'Δ GDPO_SFT-SFT':>15}")
print("="*95)
prev_g = None
for aa in aa_order:
    g = AA_TO_GROUP.get(aa, "?")
    if g != prev_g:
        print(f"  ── {g.replace(chr(10),' ')} ──")
        prev_g = g
    sft_v = freqs_all["SFT"][aa]
    dpo_v = freqs_all["DPO"][aa]
    s1    = step1[aa]
    s2    = step2[aa]
    cu    = cumul[aa]
    ab    = ablat[aa]
    flag  = ""
    if abs(cu) > 0.5:   flag += " ◀ big shift"
    if s1 * s2 < 0:     flag += " ⚠ reversal"  # DPO and GDPO pulled in opposite directions
    print(f"  {aa:<4} {'':<26} {sft_v:>7.3f} {dpo_v:>7.3f}  "
          f"{s1:>+10.3f}  {s2:>+11.3f}  {cu:>+11.3f}  {ab:>+15.3f}  {flag}")

print("="*95)

# ── Hydrophobic group summary ──────────────────────────────────────────────
hydro_aas = list("ILFVAW")
print("\n── Hydrophobic (ILFVAW) group totals ──")
print(f"  {'Model':<14} {'Sum %':>8}")
for m in ["SFT", "DPO", "GDPO", "GDPO_SFT"]:
    s = sum(freqs_all[m][aa] for aa in hydro_aas)
    print(f"  {m:<14} {s:>8.3f}")

print(f"\n  DPO  step:  {sum(step1[aa] for aa in hydro_aas):+.3f}%  ← DPO's contribution")
print(f"  GDPO step:  {sum(step2[aa] for aa in hydro_aas):+.3f}%  ← GDPO's contribution (on top of DPO)")
print(f"  GDPO_SFT:   {sum(ablat[aa] for aa in hydro_aas):+.3f}%  ← ablation (direct from SFT)")

print("\nSaved → figures/fig_aa_cascade_trace.png")


## 7b. AA Cascade Validation — DPO path at temp=0.9
Confirm the W/I hydrophobic enrichment and D→E swap in the **DPO cascade** (SFT→DPO→GDPO) are temperature-independent.
**Note:** GDPO_SFT was a single ablation run (seed=42, temp=0.7 only) — it is excluded from this temperature comparison and cited as a single-condition result.

In [ ]:

# ── Validation at temp=0.9 — DPO path only (SFT/DPO/GDPO have temp=0.9 runs)
# NOTE: GDPO_SFT was a single ablation run (seed42/temp0.7 only).
#       Validating temperature robustness for the DPO cascade is sufficient;
#       GDPO_SFT direction is confirmed from the temp=0.7 analysis above.

SEED_V, TEMP_V = 42, 0.9

# Check which models actually have temp=0.9 data
available_09 = []
missing_09   = []
for m in ["SFT", "DPO", "GDPO", "GDPO_SFT"]:
    p = BASE / m / "properties" / f"{m}_seed{SEED_V}_temp{TEMP_V}_profiled.csv"
    if p.exists():
        available_09.append(m)
    else:
        missing_09.append(m)

print(f"Available at temp=0.9 : {available_09}")
print(f"Missing  at temp=0.9  : {missing_09}")
if missing_09:
    print(f"  → {missing_09} will be excluded from the temp=0.9 comparison.\n")

key_check = ["W", "I", "F", "S", "G", "D", "N", "E"]
prop_map  = {"W":"hydrophobic","I":"hydrophobic","F":"hydrophobic",
             "S":"hydrophilic","G":"hydrophilic",
             "D":"liability↓", "N":"liability↓", "E":"charge swap"}

# ── DPO path temperature validation ───────────────────────────────────────
if all(m in available_09 for m in ["SFT", "DPO", "GDPO"]):
    freqs_v = {m: aa_freq(get_seqs(m, seed=SEED_V, temp=TEMP_V))
               for m in ["SFT", "DPO", "GDPO"]}

    step1_v = {aa: freqs_v["DPO"][aa]  - freqs_v["SFT"][aa] for aa in AAs}
    step2_v = {aa: freqs_v["GDPO"][aa] - freqs_v["DPO"][aa] for aa in AAs}

    print("DPO-path cascade validation (temp=0.7 vs temp=0.9):\n")
    print(f"  {'AA':<4}  {'Property':<14}  "
          f"{'GDPO-DPO @0.7':>14}  {'GDPO-DPO @0.9':>14}  {'|diff|':>8}  {'Same sign?':>11}")
    print("  " + "-"*80)

    all_ok = True
    for aa in key_check:
        s2_07 = step2[aa]
        s2_09 = step2_v[aa]
        diff  = abs(s2_09 - s2_07)
        ok    = (s2_07 >= 0) == (s2_09 >= 0)
        all_ok = all_ok and ok
        print(f"  {aa:<4}  {prop_map[aa]:<14}  "
              f"{s2_07:>+14.3f}  {s2_09:>+14.3f}  {diff:>8.3f}  "
              f"{'✓' if ok else '✗ CHANGED':>11}")

    print()
    print("  Hydrophobic (ILFVAW) totals:")
    print(f"  {'Model':<10} {'@0.7':>8}  {'@0.9':>8}  {'diff':>8}")
    for m in ["SFT", "DPO", "GDPO"]:
        s07 = sum(freqs_all[m][aa] for aa in hydro_aas)
        s09 = sum(freqs_v[m][aa]   for aa in hydro_aas)
        print(f"  {m:<10} {s07:>8.3f}  {s09:>8.3f}  {s09-s07:>+8.3f}")

    print(f"\n  DPO  step — @0.7: {sum(step1[aa]   for aa in hydro_aas):+.3f}%  "
          f"@0.9: {sum(step1_v[aa] for aa in hydro_aas):+.3f}%")
    print(f"  GDPO step — @0.7: {sum(step2[aa]   for aa in hydro_aas):+.3f}%  "
          f"@0.9: {sum(step2_v[aa] for aa in hydro_aas):+.3f}%")

    print()
    if all_ok:
        print("  ✓ ALL DPO-path residues show the same directional shift at temp=0.9.")
        print("  Hydrophobic enrichment (W, I, F) and D→E swap are temperature-independent.")
    else:
        print("  ✗ Some residues changed direction — check flagged rows above.")

    print()
    print("  GDPO_SFT note: single ablation run (seed42/temp0.7 only).")
    print("  GDPO_SFT direction confirmed at temp=0.7; no temp=0.9 run was performed.")
    print("  The SFT→GDPO_SFT hydrophilic enrichment (S, G) is reported as a")
    print("  single-condition observation consistent with the overall ablation finding.")

else:
    print(f"Cannot validate DPO path at temp=0.9 — missing: "
          f"{[m for m in ['SFT','DPO','GDPO'] if m not in available_09]}")
    print("Run inference for these models at temp=0.9 first.")


## 8. Per-Position Shannon Entropy
Sequence diversity at each position — shows CDR variability and GDPO's single-position convergence.

In [ ]:
def shannon_entropy_per_position(seqs, length=121):
    entropy = []
    for pos in range(length):
        aas = [s[pos] for s in seqs if len(s) > pos]
        counts = Counter(aas)
        total = sum(counts.values())
        H = -sum((c/total)*np.log2(c/total) for c in counts.values() if c > 0)
        entropy.append(H)
    return np.array(entropy)

# CDR regions (Kabat numbering for VHH, 121 AA variable region)
CDR_REGIONS = [(24, 36), (48, 64), (96, 114)]

entropy_models = ["SFT", "DPO", "GDPO","GDPO_SFT"]
fig, ax = plt.subplots(figsize=(14, 5))

MODEL_LABELS = {
    "SFT":      "SFT",
    "DPO":      "DPO",
    "GDPO":     "GDPO (DPO)",
    "GDPO_SFT": "GDPO (SFT)",
}

for m in entropy_models:
    seqs = ablation_df[ablation_df["model"]==m]["generated_sequence"].tolist()
    # trim to 121 AA variable region (remove GGGGS linker last 5)
    seqs_var = [s[:121] for s in seqs if len(s) >= 121]
    H = shannon_entropy_per_position(seqs_var, 121)
    ax.plot(range(1, 122), H, color=MODEL_COLORS[m], label=MODEL_LABELS[m],
            linewidth=1.5, alpha=0.9)

# CDR shading
for i, (start, end) in enumerate(CDR_REGIONS):
    ax.axvspan(start, end, alpha=0.12, color="steelblue")
    ax.text((start+end)/2, 3.6, f"CDR{i+1}", ha="center", fontsize=12, color="steelblue")

ax.set_xlabel("Position")
ax.set_ylabel("Shannon Entropy (bits)")
ax.legend(fontsize=9)
ax.set_xlim(1, 121)
ax.set_ylim(0, 3.8)

plt.tight_layout()
plt.savefig("figures/fig8_shannon_entropy.png", dpi=150)
plt.show()

In [ ]:
# ── CDR1 Amino Acid Frequency Analysis (positions 24-36) ─────────────────
# For each position in CDR1, show what % of sequences have each AA per model.
# Reveals exactly which amino acids GDPO converges on vs SFT diversity.

from collections import Counter

CDR1_START, CDR1_END = 24, 36   # inclusive, 1-indexed (Kabat VHH)
AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")  # standard 20 AAs

aa_models = ["SFT", "DPO", "GDPO", "GDPO_SFT"]
_t = REPORT_TEMP

# ── 1. Build per-position AA frequency tables ──────────────────────────────
# Shape: model → position → {AA: fraction}
freq_tables = {}
for m in aa_models:
    seqs = (valid_all[(valid_all["model"] == m) & (valid_all["temperature"] == _t)]
            ["generated_sequence"].tolist())
    seqs_var = [s[:121] for s in seqs if len(s) >= 121]
    tbl = {}
    for pos in range(CDR1_START - 1, CDR1_END):   # 0-indexed
        aas = [s[pos] for s in seqs_var]
        counts = Counter(aas)
        total = sum(counts.values())
        tbl[pos + 1] = {aa: counts.get(aa, 0) / total * 100 for aa in AA_ORDER}
    freq_tables[m] = tbl

positions = list(range(CDR1_START, CDR1_END + 1))

# ── 2. Heatmap: 4 models side by side ─────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
cmap = plt.cm.YlOrRd

for ax, m in zip(axes, aa_models):
    mat = np.array([[freq_tables[m][p].get(aa, 0) for p in positions]
                    for aa in AA_ORDER])   # shape (20, 13)
    im = ax.imshow(mat, aspect="auto", cmap=cmap, vmin=0, vmax=80,
                   interpolation="nearest")
    ax.set_xticks(range(len(positions)))
    ax.set_xticklabels(positions, fontsize=12)
    ax.set_title(MODEL_LABELS[m], fontsize=12, fontweight="bold",
                 color=MODEL_COLORS[m])
    ax.set_xlabel("CDR1 Position",fontsize=10)
    if ax is axes[0]:
        ax.set_yticks(range(len(AA_ORDER)))
        ax.set_yticklabels(AA_ORDER, fontsize=10)

fig.colorbar(im, ax=axes[-1], label="% of sequences", shrink=0.8)
plt.tight_layout()
plt.savefig("figures/fig_cdr1_aa_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

# ── 3. Top AAs at entropy-diverging positions (print table) ───────────────
# Compute entropy delta: H(GDPO) - H(SFT) per CDR1 position
def pos_entropy(freq_dict):
    vals = np.array(list(freq_dict.values())) / 100.0
    vals = vals[vals > 0]
    return -np.sum(vals * np.log2(vals))

print(f"\nCDR1 AA summary — top-3 AAs per position (T={_t})")
print(f"{'Pos':>4}  {'ΔH(GDPO-SFT)':>14}  " +
      "  ".join(f"{MODEL_LABELS[m]:>22}" for m in aa_models))
print("-" * (4 + 16 + 26 * len(aa_models)))

for p in positions:
    h_sft  = pos_entropy(freq_tables["SFT"][p])
    h_gdpo = pos_entropy(freq_tables["GDPO"][p])
    delta  = h_gdpo - h_sft

    def top3(m):
        d = freq_tables[m][p]
        top = sorted(d.items(), key=lambda x: -x[1])[:3]
        return "  ".join(f"{aa}:{v:.0f}%" for aa, v in top if v > 1)

    row = f"{p:>4}  {delta:>+14.3f}  " + "  ".join(f"{top3(m):>22}" for m in aa_models)
    print(row)

# ── 4. Bar chart: top-5 AAs at the 3 most diverging positions ─────────────
entropies_sft  = np.array([pos_entropy(freq_tables["SFT"][p])  for p in positions])
entropies_gdpo = np.array([pos_entropy(freq_tables["GDPO"][p]) for p in positions])
delta_arr = entropies_gdpo - entropies_sft
top3_pos_idx = np.argsort(np.abs(delta_arr))[::-1][:3]
top3_pos = [positions[i] for i in top3_pos_idx]

fig, axes = plt.subplots(1, len(top3_pos), figsize=(6 * len(top3_pos), 5))
if len(top3_pos) == 1:
    axes = [axes]

for ax, p in zip(axes, top3_pos):
    # union of top-5 AAs across all models at this position
    all_aas = set()
    for m in aa_models:
        top5 = sorted(freq_tables[m][p].items(), key=lambda x: -x[1])[:5]
        all_aas.update(aa for aa, v in top5 if v > 1)
    aa_list = sorted(all_aas, key=lambda aa: -max(freq_tables[m][p].get(aa, 0)
                                                    for m in aa_models))[:8]

    x = np.arange(len(aa_list))
    w = 0.2
    for i, m in enumerate(aa_models):
        vals = [freq_tables[m][p].get(aa, 0) for aa in aa_list]
        ax.bar(x + (i - 1.5) * w, vals, w, color=MODEL_COLORS[m],
               label=MODEL_LABELS[m], alpha=0.85, edgecolor="white")

    ax.set_xticks(x)
    ax.set_xticklabels(aa_list, fontsize=10)
    ax.set_ylabel("Number of sequences(%)")
    dh = entropies_gdpo[positions.index(p)] - entropies_sft[positions.index(p)]
    # ax.set_title(f"Position {p}  (ΔH={dh:+.2f} bits)", fontsize=11)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 100)

# plt.suptitle(f"CDR1 — AA composition at highest-divergence positions  (T={_t})",
#              fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("figures/fig_cdr1_aa_divergence.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ── CDR2 & CDR3 Amino Acid Frequency Analysis ────────────────────────────
# Same analysis as CDR1: per-position AA frequencies + entropy delta + bar charts
# CDR2: 48-64,  CDR3: 96-114  (Kabat VHH numbering)

from collections import Counter

CDR_DEFS = {
    "CDR2": (48, 64),
    "CDR3": (96, 114),
}
AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")
aa_models = ["SFT", "DPO", "GDPO", "GDPO_SFT"]
_t = REPORT_TEMP

def pos_entropy(freq_dict):
    vals = np.array(list(freq_dict.values())) / 100.0
    vals = vals[vals > 0]
    return -np.sum(vals * np.log2(vals))

def build_freq_table(start, end):
    tables = {}
    for m in aa_models:
        seqs = (valid_all[(valid_all["model"] == m) & (valid_all["temperature"] == _t)]
                ["generated_sequence"].tolist())
        seqs_var = [s[:121] for s in seqs if len(s) >= 121]
        tbl = {}
        for pos in range(start - 1, end):   # 0-indexed
            aas = [s[pos] for s in seqs_var]
            counts = Counter(aas)
            total = sum(counts.values())
            tbl[pos + 1] = {aa: counts.get(aa, 0) / total * 100 for aa in AA_ORDER}
        tables[m] = tbl
    return tables

for cdr_name, (cdr_start, cdr_end) in CDR_DEFS.items():
    positions = list(range(cdr_start, cdr_end + 1))
    freq_tables = build_freq_table(cdr_start, cdr_end)

    # ── Heatmap ────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=True)
    cmap = plt.cm.YlOrRd
    for ax, m in zip(axes, aa_models):
        mat = np.array([[freq_tables[m][p].get(aa, 0) for p in positions]
                        for aa in AA_ORDER])
        im = ax.imshow(mat, aspect="auto", cmap=cmap, vmin=0, vmax=80,
                       interpolation="nearest")
        ax.set_xticks(range(len(positions)))
        ax.set_xticklabels(positions, fontsize=12, rotation=45)
        ax.set_title(MODEL_LABELS[m], fontsize=12, fontweight="bold",
                     color=MODEL_COLORS[m])
        ax.set_xlabel(f"{cdr_name} Position",fontsize=10)
        if ax is axes[0]:
            ax.set_yticks(range(len(AA_ORDER)))
            ax.set_yticklabels(AA_ORDER, fontsize=10)
    fig.colorbar(im, ax=axes[-1], label="% of sequences", shrink=0.8)
    plt.tight_layout()
    plt.savefig(f"figures/fig_{cdr_name.lower()}_aa_heatmap.png", dpi=300, bbox_inches="tight")
    plt.show()

    # ── Text summary table ─────────────────────────────────────────────────
    print(f"\n{cdr_name} AA summary — top-3 AAs per position (T={_t})")
    print(f"{'Pos':>4}  {'ΔH(GDPO-SFT)':>14}  " +
          "  ".join(f"{MODEL_LABELS[m]:>22}" for m in aa_models))
    print("-" * (4 + 16 + 26 * len(aa_models)))
    for p in positions:
        h_sft  = pos_entropy(freq_tables["SFT"][p])
        h_gdpo = pos_entropy(freq_tables["GDPO"][p])
        delta  = h_gdpo - h_sft
        def top3(m):
            d = freq_tables[m][p]
            top = sorted(d.items(), key=lambda x: -x[1])[:3]
            return "  ".join(f"{aa}:{v:.0f}%" for aa, v in top if v > 1)
        row = f"{p:>4}  {delta:>+14.3f}  " + "  ".join(f"{top3(m):>22}" for m in aa_models)
        print(row)

    # ── Bar chart: 3 most diverging positions ──────────────────────────────
    entropies_sft  = np.array([pos_entropy(freq_tables["SFT"][p])  for p in positions])
    entropies_gdpo = np.array([pos_entropy(freq_tables["GDPO"][p]) for p in positions])
    delta_arr = entropies_gdpo - entropies_sft

    # Top 3 by absolute delta, skip fully conserved (|delta| < 0.01)
    candidate_idx = np.argsort(np.abs(delta_arr))[::-1]
    top3_pos_idx  = [i for i in candidate_idx if abs(delta_arr[i]) > 0.01][:3]
    top3_pos = [positions[i] for i in top3_pos_idx]

    if top3_pos:
        fig, axes = plt.subplots(1, len(top3_pos), figsize=(6 * len(top3_pos), 5))
        if len(top3_pos) == 1:
            axes = [axes]
        for ax, p in zip(axes, top3_pos):
            all_aas = set()
            for m in aa_models:
                top5 = sorted(freq_tables[m][p].items(), key=lambda x: -x[1])[:5]
                all_aas.update(aa for aa, v in top5 if v > 1)
            aa_list = sorted(all_aas,
                             key=lambda aa: -max(freq_tables[m][p].get(aa, 0)
                                                  for m in aa_models))[:8]
            x  = np.arange(len(aa_list))
            w  = 0.2
            for i, m in enumerate(aa_models):
                vals = [freq_tables[m][p].get(aa, 0) for aa in aa_list]
                ax.bar(x + (i - 1.5) * w, vals, w, color=MODEL_COLORS[m],
                       label=MODEL_LABELS[m], alpha=0.85, edgecolor="white")
            ax.set_xticks(x)
            ax.set_xticklabels(aa_list, fontsize=10)
            ax.set_ylabel("Number of sequences(%)")
            dh = delta_arr[positions.index(p)]
            # ax.set_title(f"Position {p}  (ΔH={dh:+.2f} bits)", fontsize=11)
            ax.legend(fontsize=8)
            ax.set_ylim(0, 100)
        # plt.suptitle(f"{cdr_name} — AA composition at highest-divergence positions  (T={_t})",
        #              fontsize=12, fontweight="bold", y=1.02)
        plt.tight_layout()
        plt.savefig(f"figures/fig_{cdr_name.lower()}_aa_divergence.png",
                    dpi=300, bbox_inches="tight")
        plt.show()
    else:
        print(f"  [{cdr_name}] All positions fully conserved — no divergence bar chart needed.")

In [ ]:
# ── Global AA composition shift — enrichment relative to SFT ─────────────
# Shannon entropy shows WHERE. This shows WHAT replaced the liability residues.

from collections import Counter

_t        = REPORT_TEMP
aa_models = ["SFT", "DPO", "GDPO", "GDPO_SFT"]
AA_ORDER  = list("ACDEFGHIKLMNPQRSTVWY")

def global_aa_freq(model, temp):
    seqs = (valid_all[(valid_all["model"] == model) & (valid_all["temperature"] == temp)]
            ["generated_sequence"].tolist())
    seqs_var = [s[:121] for s in seqs if len(s) >= 121]
    all_aa   = "".join(seqs_var)
    total    = len(all_aa)
    return {aa: all_aa.count(aa) / total * 100 for aa in AA_ORDER}

def pos_aa_freq(model, temp, position):
    seqs = (valid_all[(valid_all["model"] == model) & (valid_all["temperature"] == temp)]
            ["generated_sequence"].tolist())
    seqs_var = [s[:121] for s in seqs if len(s) >= 121]
    aas   = [s[position - 1] for s in seqs_var]
    total = len(aas)
    return {aa: aas.count(aa) / total * 100 for aa in AA_ORDER}

global_freqs = {m: global_aa_freq(m, _t) for m in aa_models}

# Frequency ratio vs SFT: (model_freq / SFT_freq) - 1  → 0 = same as SFT
# Positive = enriched, negative = depleted relative to SFT
ratios = {m: {aa: (global_freqs[m][aa] / global_freqs["SFT"][aa]) - 1.0
              if global_freqs["SFT"][aa] > 0 else 0.0
              for aa in AA_ORDER}
          for m in ["DPO", "GDPO", "GDPO_SFT"]}

# ── Ordered: hydrophobic → aromatic → polar → charged → small ─────────────
# Arranged left-to-right by hydrophobicity story
ORDERED_AAS = ["I", "V", "L", "A",   # aliphatic hydrophobic
               "Y", "F", "W",          # aromatic
               "S", "T", "Q",          # polar (S = GDPO_SFT signature)
               "R", "E",               # charged divergence
               "G", "M"]               # structural / oxidation

# Soft background bands — no text labels, just shading
BANDS = [
    (0,  3,  "#4C72B0", "Hydrophobic"),
    (4,  6,  "#8172B2", "Aromatic"),
    (7,  9,  "#64B5CD", "Polar"),
    (10, 11, "#55A868", "Charged"),
    (12, 13, "#BCB08A", "Structural"),
]

plot_models = ["DPO", "GDPO", "GDPO_SFT"]
n_models    = len(plot_models)
w           = 0.22
x           = np.arange(len(ORDERED_AAS))

fig, ax = plt.subplots(figsize=(13, 5))

for i, m in enumerate(plot_models):
    vals   = [ratios[m][aa] for aa in ORDERED_AAS]
    offset = (i - (n_models - 1) / 2) * w
    ax.bar(x + offset, vals, w,
           color=MODEL_COLORS[m], alpha=0.88,
           edgecolor="white", linewidth=0.5,
           label=MODEL_LABELS[m])

ax.axhline(0, color="black", linewidth=1.0, linestyle="--", alpha=0.7)

# subtle band shading — no labels on plot
for (s, e, col, _) in BANDS:
    ax.axvspan(s - 0.5, e + 0.5, alpha=0.07, color=col, zorder=0)
    # thin separator between bands
    if e < len(ORDERED_AAS) - 1:
        ax.axvline(e + 0.5, color="grey", linewidth=0.6, linestyle=":", alpha=0.4)

ax.set_xticks(x)
ax.set_xticklabels(ORDERED_AAS, fontsize=13, fontweight="bold")
ax.set_ylabel("Frequency ratio w/SFT", fontsize=10)
ax.legend(fontsize=10, framealpha=0.9, loc="upper left")
ax.set_xlim(-0.6, len(ORDERED_AAS) - 0.4)

# band legend (small, bottom right)
from matplotlib.patches import Patch
band_patches = [Patch(color=col, alpha=0.35, label=lbl) for _, _, col, lbl in BANDS]
ax.legend(
    handles=[plt.Rectangle((0,0),1,1, color=MODEL_COLORS[m], alpha=0.88,
                             label=MODEL_LABELS[m]) for m in plot_models],
    fontsize=10, framealpha=0.9, loc="upper left"
)

plt.tight_layout()
plt.savefig("figures/fig3_global_aa_shift.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figures/fig3_global_aa_shift.png")

# ── CDR key positions ─────────────────────────────────────────────────────
KEY_POSITIONS = {
    "CDR1": [27, 28, 30],
}
STORY = {
    27: "D→A\n(isomerization risk)",
    28: "G→R\n(flexibility→rigidity)",
    30: "N→I/F\n(DPO liability corrected)",
    55: "→S (converged)",
    56: "→W (converged)",
    57: "→G (converged)",
}

for cdr_name, pos_list in KEY_POSITIONS.items():
    n_pos = len(pos_list)
    fig_b, axes_b = plt.subplots(1, n_pos, figsize=(5.5 * n_pos, 5), sharey=False)
    if n_pos == 1:
        axes_b = [axes_b]

    for ax, p in zip(axes_b, pos_list):
        freqs = {m: pos_aa_freq(m, _t, p) for m in aa_models}
        all_aas = set()
        for m in aa_models:
            top5 = sorted(freqs[m].items(), key=lambda x: -x[1])[:5]
            all_aas.update(aa for aa, v in top5 if v > 1)
        aa_list = sorted(all_aas,
                         key=lambda aa: -max(freqs[m].get(aa, 0) for m in aa_models))[:6]
        xi = np.arange(len(aa_list))
        wi = 0.2
        for i, m in enumerate(aa_models):
            vals = [freqs[m].get(aa, 0) for aa in aa_list]
            ax.bar(xi + (i - 1.5) * wi, vals, wi,
                   color=MODEL_COLORS[m], label=MODEL_LABELS[m],
                   alpha=0.85, edgecolor="white", linewidth=0.5)
        ax.set_xticks(xi)
        ax.set_xticklabels(aa_list, fontsize=11, fontweight="bold")
        ax.set_ylabel("Residue frequency (%)", fontsize=10)
        ax.set_ylim(0, 105)
        ax.set_title(f"{cdr_name} pos {p}\n{STORY.get(p, '')}",
                     fontsize=10, fontweight="bold")
        ax.legend(fontsize=8, framealpha=0.9)

    plt.tight_layout()
    fname = f"figures/fig3{'b' if cdr_name=='CDR1' else 'c'}_{cdr_name.lower()}_key_positions.png"
    plt.savefig(fname, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")


In [ ]:

# Shannon entropy at T=0.9
_t = 0.9
fig, ax = plt.subplots(figsize=(14, 5))
for m in entropy_models:
    seqs = valid_all[(valid_all["model"]==m) & (valid_all["temperature"]==_t)]["generated_sequence"].tolist()
    seqs_var = [s[:121] for s in seqs if len(s) >= 121]
    if not seqs_var:
        print(f"{m} T={_t}: no sequences")
        continue
    H = shannon_entropy_per_position(seqs_var, 121)
    ax.plot(range(1, 122), H, color=MODEL_COLORS[m], label=MODEL_LABELS[m],
            linewidth=1.5, alpha=0.9)

for i, (start, end) in enumerate(CDR_REGIONS):
    ax.axvspan(start, end, alpha=0.12, color="steelblue")
    ax.text((start+end)/2, 3.6, f"CDR{i+1}", ha="center", fontsize=12, color="steelblue")

ax.set_xlabel("Position")
ax.set_ylabel("Shannon Entropy (bits)")
ax.set_title(f"Shannon Entropy per Position  (T={_t})")
ax.legend(fontsize=9)
ax.set_xlim(1, 121)
ax.set_ylim(0, 3.8)
plt.tight_layout()
plt.savefig(f"figures/fig8_shannon_entropy_t{_t}.png", dpi=150)
plt.show()


In [ ]:

# Shannon entropy at T=1.2
_t = 1.2
fig, ax = plt.subplots(figsize=(14, 5))
for m in entropy_models:
    seqs = valid_all[(valid_all["model"]==m) & (valid_all["temperature"]==_t)]["generated_sequence"].tolist()
    seqs_var = [s[:121] for s in seqs if len(s) >= 121]
    if not seqs_var:
        print(f"{m} T={_t}: no sequences")
        continue
    H = shannon_entropy_per_position(seqs_var, 121)
    ax.plot(range(1, 122), H, color=MODEL_COLORS[m], label=MODEL_LABELS[m],
            linewidth=1.5, alpha=0.9)

for i, (start, end) in enumerate(CDR_REGIONS):
    ax.axvspan(start, end, alpha=0.12, color="steelblue")
    ax.text((start+end)/2, 3.6, f"CDR{i+1}", ha="center", fontsize=12, color="steelblue")

ax.set_xlabel("Position")
ax.set_ylabel("Shannon Entropy (bits)")
ax.set_title(f"Shannon Entropy per Position  (T={_t})")
ax.legend(fontsize=9)
ax.set_xlim(1, 121)
ax.set_ylim(0, 3.8)
plt.tight_layout()
plt.savefig(f"figures/fig8_shannon_entropy_t{_t}.png", dpi=150)
plt.show()


## 9. Cross-Seed Robustness — Tm & Liability
Confirms results are not seed-dependent. Shows mean ± std across seeds for each temperature.

In [ ]:
main_models = ["SFT", "DPO", "GDPO"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (col, label) in zip(axes, [("tempro_tm","Tm (°C)"), ("liability_severity","Liability Severity")]):
    for m in main_models:
        d = (
            valid_all[valid_all["model"]==m]
            .groupby(["seed","temperature"])[col].mean()
            .reset_index()
        )
        # mean and std across seeds for each temp
        by_temp = d.groupby("temperature")[col].agg(["mean","std"]).reset_index()
        ax.errorbar(by_temp["temperature"], by_temp["mean"], yerr=by_temp["std"],
                    label=MODEL_LABELS[m], color=MODEL_COLORS[m],
                    marker="o", linewidth=2, capsize=5)

    ax.set_xlabel("Temperature")
    ax.set_ylabel(label)
    ax.set_title(f"{label} — mean ± std across 3 seeds")
    ax.set_xticks(TEMPS)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("figures/fig9_cross_seed_robustness.png", dpi=150)
plt.show()

## 10. Full Summary Table — All Properties, All Models
Single reference table for the paper.

In [ ]:
summary_cols = {
    "tempro_tm":           "Tm (°C)",
    "instability_index":   "Instability Index",
    "sapiens_humanness":   "Humanness",
    "liability_severity":  "Liability Severity",
    "gravy":               "GRAVY",
    "isoelectric_point":   "pI",
    "charge_at_pH7":       "Charge pH7",
    "expression_score":    "Expression",
    "aggrescan_nhs":       "Aggrescan NHs",
    "is_valid_126":        "Validity (%)",
}

rows = []
for m in MODELS:
    if m not in valid_all["model"].unique() and m != "GDPO_SFT":
        continue
    df_m = valid_all[valid_all["model"]==m] if m != "GDPO_SFT" else ablation_df[ablation_df["model"]==m]
    row = {"Model": MODEL_LABELS[m]}
    for col, label in summary_cols.items():
        if col not in df_m.columns:
            row[label] = "N/A"
        elif col == "is_valid_126":
            run_vals = df_m.groupby(["seed","temperature"])[col].mean().mul(100)
            row[label] = f"{run_vals.mean():.1f} ± {run_vals.std():.1f}"
        else:
            run_vals = df_m.groupby(["seed","temperature"])[col].mean()
            row[label] = f"{run_vals.mean():.3f} ± {run_vals.std():.3f}"
    rows.append(row)

summary_table = pd.DataFrame(rows).set_index("Model")
print(summary_table.T.to_string())
summary_table.T.to_csv("figures/summary_table.csv")

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 8,
})
SEED_R = 42
TEMP_R = 0.7

RADAR_METRICS = [
    ("tempro_tm",           "Thermal Stability",       60,   85,   False),
    # ("sapiens_humanness",   "Humanness",     0.5, 0.75, False),
    ("instability_index",   "Structural Stability",     25,   45,   True),
    # ("expression_score",    "Expression",    0, 1.00, False),
    ("liability_severity",  "Chemical Stability", 5,    16,   True),
    ("netsolp_solubility",  "Solubility",    0.4, 0.8, False), # 
     ("netsolp_usability",   "Expression",    0.3, 0.50, False),
]

def _norm(val, rmin, rmax, inv):
    if pd.isna(val):
        return np.nan
    v = np.clip((val - rmin) / (rmax - rmin), 0, 1)
    return 1 - v if inv else v

# Muted publication-style palette
COLORS = {
    "SFT":      "#2D6FA5",   # muted steel blue
    "DPO":      "#C05C26",   # terracotta
    "GDPO":     "#2E7A50",   # deep green
    "GDPO_SFT": "#7A3B5E",   # muted plum
}

DISPLAY_NAMES = {
    "SFT":      "SFT",
    "DPO":      "DPO",
    "GDPO":     "SFT→DPO→GDPO",
    "GDPO_SFT": "SFT→GDPO",
}
# ── Load data ─────────────────────────────────────────────────────────────
radar_data = {}
for m in ["SFT", "DPO", "GDPO", "GDPO_SFT"]:
    p = BASE / m / "properties" / f"{m}_seed{SEED_R}_temp{TEMP_R}_profiled.csv"
    if p.exists():
        radar_data[m] = pd.read_csv(p)

if not radar_data:
    raise FileNotFoundError("No radar CSV files found. Check BASE / paths.")

N = len(RADAR_METRICS)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

# ── Figure ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6.6, 6.6), subplot_kw=dict(polar=True))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# First axis at top, clockwise
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_ylim(0, 1.0)

# Turn off all default polar grid/spine elements
ax.grid(False)
ax.xaxis.grid(False)
ax.yaxis.grid(False)
ax.spines["polar"].set_visible(False)
ax.set_yticks([])

# ── Custom polygon grid ───────────────────────────────────────────────────
ring_radii = [0.25, 0.50, 0.75, 1.00]

# polygon rings
for r in ring_radii:
    ax.plot(
        angles,
        [r] * (N + 1),
        color="#ececec",
        linewidth=0.55 if r < 1.0 else 0.70,
        linestyle="-",
        zorder=0,
    )

# spokes clipped exactly to outer hexagon
for a in angles[:-1]:
    ax.plot([a, a], [0, 1.0], color="#efefef", linewidth=0.5, zorder=0)

# radial percentage labels on top spoke
label_angle = angles[0]
for r in ring_radii:
    offset = 0.010 if r < 1.0 else 0.006
    ax.text(
        label_angle,
        r + offset,
        f"{int(r * 100)}%",
        ha="center",
        va="bottom",
        fontsize=10,
        color="#b3b3b3",
    )

# ── Model polygons ────────────────────────────────────────────────────────
for m, df in radar_data.items():
    vals = []
    for col, _, rmin, rmax, inv in RADAR_METRICS:
        if col in df.columns and df[col].notna().any():
            vals.append(_norm(df[col].mean(), rmin, rmax, inv))
        else:
            vals.append(np.nan)

    vc = vals + vals[:1]

    ax.plot(
        angles,
        vc,
        color=COLORS[m],
        linewidth=1.9,
        zorder=3,
        label=DISPLAY_NAMES[m],
        solid_capstyle="round",
        solid_joinstyle="round",
    )
    ax.fill(angles, vc, color=COLORS[m], alpha=0.09, zorder=2)
    ax.scatter(
        angles[:-1],
        vals,
        color=COLORS[m],
        s=26,
        zorder=4,
        edgecolors="white",
        linewidths=1.2,
    )

# ── Axis labels ───────────────────────────────────────────────────────────
ax.set_xticks(angles[:-1])
ax.set_xticklabels(
    [m[1] for m in RADAR_METRICS],
    fontsize=12,
    color="#222222",
    fontweight="normal",
)
ax.tick_params(axis="x", pad=16)

# ── Legend ────────────────────────────────────────────────────────────────
ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.18),
    ncol=2,
    frameon=False,
    fontsize=12,
    handlelength=1.4,
    columnspacing=1.0,
    handletextpad=0.4,
)

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig(
    "figures/radar_all_models.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()

# ── Print normalized values ───────────────────────────────────────────────
print("\nNorm values (0→1, higher=better):")
hdr = f"  {'Model':<30}" + "".join(f"{m[1]:>16}" for m in RADAR_METRICS)
print(hdr)
print("  " + "-" * (30 + 16 * N))

for m, df in radar_data.items():
    vals = []
    for col, _, rmin, rmax, inv in RADAR_METRICS:
        if col in df.columns and df[col].notna().any():
            vals.append(_norm(df[col].mean(), rmin, rmax, inv))
        else:
            vals.append(np.nan)

    print(f"  {DISPLAY_NAMES[m]:<30}" + "".join(f"{v:>16.3f}" for v in vals))

print("\nSaved → figures/radar_all_models.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

# Data points
models = {
    'SFT': (13.4, 71.8),
    'DPO': (12.5, 72.8),
    'GDPO': (7.9, 77.9),
    'GDPO-direct': (7.9, 76.0)
}

# Create fitness landscape (example: higher Tm + lower liability = better)
x = np.linspace(0, 16, 100)
y = np.linspace(68, 82, 100)
X, Y = np.meshgrid(x, y)

# Fitness function (example: gaussian peaks at ideal regions)
fitness = np.exp(-((X-7)**2 + (Y-78)**2)/20) * 100

fig, ax = plt.subplots(figsize=(8, 6))

# Contour plot (fitness landscape)
contours = ax.contourf(X, Y, fitness, levels=15, cmap='RdYlGn', alpha=0.6)
ax.contour(X, Y, fitness, levels=15, colors='gray', alpha=0.3, linewidths=0.5)

# Plot trajectories
# SFT → DPO → GDPO
ax.annotate('', xy=(12.5, 72.8), xytext=(13.4, 71.8),
            arrowprops=dict(arrowstyle='->', lw=2, color='#2E86AB'))
ax.annotate('', xy=(7.9, 77.9), xytext=(12.5, 72.8),
            arrowprops=dict(arrowstyle='->', lw=2, color='#2E86AB'))

# SFT → GDPO-direct
ax.annotate('', xy=(7.9, 76.0), xytext=(13.4, 71.8),
            arrowprops=dict(arrowstyle='->', lw=2, color='#A23B72', linestyle='--'))

# Plot points
ax.plot(13.4, 71.8, 'o', markersize=12, color='#5A8AC6', label='SFT', zorder=5)
ax.plot(12.5, 72.8, 'D', markersize=12, color='#E89C4F', label='DPO', zorder=5)
ax.plot(7.9, 77.9, '*', markersize=18, color='#6ABE6E', label='GDPO (via DPO)', zorder=5)
ax.plot(7.9, 76.0, 's', markersize=12, color='#D95F5F', label='GDPO-direct', zorder=5)

ax.set_xlabel('Chemical Liability Severity', fontsize=12)
ax.set_ylabel('Thermal Stability (°C)', fontsize=12)
ax.set_title('Optimization Landscape Trajectories', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()



In [ ]:
# Figure 4 — Optimization landscape: all targets, all models
# Each point = per-target mean (1 pt per target×model, ~244 pts total)
# Dashed ellipses = 1.5σ covariance spread of each model across all targets

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

os.makedirs('figures', exist_ok=True)
_t, _seed = 0.7, 42
aa_models = ['SFT', 'DPO', 'GDPO', 'GDPO_SFT']

data42 = valid_all[
    (valid_all['seed'] == _seed) & (valid_all['temperature'] == _t)
].copy()

def is_valid_seq(seq):
    if not isinstance(seq, str): return False
    return len(seq) == 126 and seq.endswith('GGGGS') and seq.count('C') == 2

data42 = data42[data42['generated_sequence'].apply(is_valid_seq)].copy()
n_seqs = data42.groupby(['peptide', 'model']).size().unstack(fill_value=0)
valid_peps = n_seqs[(n_seqs >= 20).all(axis=1)].index
data42_filt = data42[data42['peptide'].isin(valid_peps)]

g = (
    data42_filt
    .groupby(['peptide', 'model'])[['tempro_tm', 'liability_severity']]
    .mean()
    .reset_index()
)

def confidence_ellipse(x, y, ax, n_std=1.5, **kwargs):
    if len(x) < 3: return
    cov = np.cov(x, y)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h  = 2 * n_std * np.sqrt(vals)
    ell   = Ellipse(xy=(np.mean(x), np.mean(y)), width=w, height=h, angle=angle, **kwargs)
    ax.add_patch(ell)

MARKERS = {'SFT': 'o', 'DPO': 'D', 'GDPO': '*', 'GDPO_SFT': 's'}
SIZES   = {'SFT': 45,  'DPO': 45,  'GDPO': 95,  'GDPO_SFT': 45}

fig, ax = plt.subplots(figsize=(10, 7))

for m in aa_models:
    sub = g[g['model'] == m]
    x, y = -sub['liability_severity'].values, sub['tempro_tm'].values

    # Filled ellipse (light)
    confidence_ellipse(x, y, ax, n_std=1.5,
                       facecolor=MODEL_COLORS[m], alpha=0.12, zorder=1)
    # Ellipse outline
    confidence_ellipse(x, y, ax, n_std=1.5, facecolor='none',
                       edgecolor=MODEL_COLORS[m], linewidth=2.0,
                       linestyle='--', alpha=0.85, zorder=2)

    # Per-target mean dots
    ax.scatter(x, y, color=MODEL_COLORS[m], marker=MARKERS[m],
               s=SIZES[m], alpha=0.55, edgecolors='none',
               label=f'{MODEL_LABELS[m]}', zorder=3)

    # Overall centroid (white-outlined)
    ax.scatter(x.mean(), y.mean(), color=MODEL_COLORS[m],
               marker=MARKERS[m], s=SIZES[m] * 3.5,
               edgecolors='white', linewidths=2.0, zorder=6)

ax.set_xlabel('\u2190 Higher Liability          Mean Liability Severity per target          Lower Liability (better) \u2192', fontsize=11)
ax.set_ylabel('Mean Tm per target  (\u00b0C)', fontsize=12)
ax.grid(alpha=0.25, linestyle='--')
for spine in ax.spines.values():
    spine.set_visible(True)
ax.legend(fontsize=10, framealpha=0.92, loc='lower right')

plt.tight_layout()
plt.savefig('figures/fig4_landscape.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: figures/fig4_landscape.png  ({len(valid_peps)} targets)')


In [ ]:
# Figure 4b — Emergent improvement: Tm vs Instability Index (not a reward metric)
# GDPO(DPO) and GDPO(SFT) separate more clearly here — shows the two paths
# lead to different structural outcomes via different amino acid routes.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

g_ii = (
    data42_filt
    .groupby(['peptide', 'model'])[['tempro_tm', 'instability_index']]
    .mean()
    .reset_index()
)

MARKERS = {'SFT': 'o', 'DPO': 'D', 'GDPO': '*', 'GDPO_SFT': 's'}
SIZES   = {'SFT': 45,  'DPO': 45,  'GDPO': 95,  'GDPO_SFT': 45}

fig, ax = plt.subplots(figsize=(10, 7))

for m in aa_models:
    sub = g_ii[g_ii['model'] == m]
    x, y = -sub['instability_index'].values, sub['tempro_tm'].values

    confidence_ellipse(x, y, ax, n_std=1.5,
                       facecolor=MODEL_COLORS[m], alpha=0.12, zorder=1)
    confidence_ellipse(x, y, ax, n_std=1.5, facecolor='none',
                       edgecolor=MODEL_COLORS[m], linewidth=2.0,
                       linestyle='--', alpha=0.85, zorder=2)

    ax.scatter(x, y, color=MODEL_COLORS[m], marker=MARKERS[m],
               s=SIZES[m], alpha=0.55, edgecolors='none',
               label=MODEL_LABELS[m], zorder=3)

    ax.scatter(x.mean(), y.mean(), color=MODEL_COLORS[m],
               marker=MARKERS[m], s=SIZES[m] * 3.5,
               edgecolors='white', linewidths=2.0, zorder=6)

ax.set_xlabel('\u2190 Higher Instability          Mean Instability Index per target          Lower Instability (better) \u2192', fontsize=11)
ax.set_ylabel('Mean Tm per target  (\u00b0C)', fontsize=12)
ax.axvline(-40, color='gray', linestyle=':', linewidth=1.2, alpha=0.6)
ax.grid(alpha=0.25, linestyle='--')
ax.legend(fontsize=10, framealpha=0.92, loc='lower right')

plt.tight_layout()
plt.savefig('figures/fig4b_instability_landscape.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figures/fig4b_instability_landscape.png')

# Print centroid comparison (GDPO vs GDPO_SFT separation)
print('\n-- Centroid instability index (lower = more stable) --')
for m in aa_models:
    sub = g_ii[g_ii['model'] == m]
    print(f'  {MODEL_LABELS[m]:12s}: II={sub["instability_index"].mean():.2f}  Tm={sub["tempro_tm"].mean():.2f}')


In [ ]:
# ── Figure 4 (3D) — HOW models achieve optimization: Tm × GRAVY × Charge ──
#
# Z = Tm (what we optimize)
# X = GRAVY (hydrophobicity — mechanistic: GDPO_DPO uses hydrophobic packing)
# Y = Charge at pH7 (electrostatic strategy — differs between the two paths)
#
# Shows not just THAT models improve, but the amino acid strategy they use.

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

_t, _seed = 0.7, 42
aa_models = ['SFT', 'DPO', 'GDPO', 'GDPO_SFT']

data42 = valid_all[
    (valid_all['seed'] == _seed) & (valid_all['temperature'] == _t)
].copy()

def is_valid_seq(seq):
    if not isinstance(seq, str): return False
    return len(seq) == 126 and seq.endswith('GGGGS') and seq.count('C') == 2

data42 = data42[data42['generated_sequence'].apply(is_valid_seq)].copy()
n_seqs = data42.groupby(['peptide', 'model']).size().unstack(fill_value=0)
valid_peps = n_seqs[(n_seqs >= 20).all(axis=1)].index
data42_filt = data42[data42['peptide'].isin(valid_peps)]

g3d = (
    data42_filt
    .groupby(['peptide', 'model'])[['tempro_tm', 'gravy', 'charge_at_pH7']]
    .mean()
    .reset_index()
)

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

def draw_ellipsoid(ax, cx, cy, cz, sx, sy, sz, color, n_std=2.0, n_pts=25):
    u = np.linspace(0, 2 * np.pi, n_pts)
    v = np.linspace(0, np.pi, n_pts)
    x = cx + n_std * sx * np.outer(np.cos(u), np.sin(v))
    y = cy + n_std * sy * np.outer(np.sin(u), np.sin(v))
    z = cz + n_std * sz * np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(x, y, z, color=color, alpha=0.12, linewidth=0.4)
    ax.plot_surface(x, y, z, color=color, alpha=0.04, shade=False)

for m in aa_models:
    sub = g3d[g3d['model'] == m]
    x = sub['gravy'].values
    y = sub['charge_at_pH7'].values
    z = sub['tempro_tm'].values

    ax.scatter(x, y, z,
               color=MODEL_COLORS[m], marker='o', s=45,
               alpha=0.50, edgecolors='none', label=MODEL_LABELS[m],
               depthshade=True)

    ax.scatter([x.mean()], [y.mean()], [z.mean()],
               color=MODEL_COLORS[m], marker='o', s=200,
               edgecolors='white', linewidths=2.0, zorder=10,
               depthshade=False)

    if len(x) >= 3:
        draw_ellipsoid(ax, x.mean(), y.mean(), z.mean(),
                       x.std(), y.std(), z.std(),
                       MODEL_COLORS[m], n_std=2.0)

ax.set_xlabel('GRAVY (hydrophobicity \u2192)', fontsize=10, labelpad=10)
ax.set_ylabel('Charge at pH 7', fontsize=10, labelpad=10)
ax.set_zlabel('Predicted Tm  (\u00b0C  \u2191)', fontsize=10, labelpad=10)

ax.set_title(
    'Figure 4 (3D) \u2014 Amino acid strategy landscape\n'
    'How each model achieves Tm: hydrophobicity vs charge balance.\n'
    'Per-target means (~61 targets). Ellipsoids = 2\u03c3. Large marker = centroid.',
    fontsize=11, fontweight='bold', pad=20)

ax.legend(fontsize=9, loc='upper left', framealpha=0.9)
ax.view_init(elev=20, azim=-55)
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False
ax.xaxis.pane.set_edgecolor('#ddd')
ax.yaxis.pane.set_edgecolor('#ddd')
ax.zaxis.pane.set_edgecolor('#ddd')
ax.grid(alpha=0.15)

print(f"{'Model':<14} {'GRAVY':>8} {'Charge':>8} {'Tm':>8}")
print('-' * 42)
for m in aa_models:
    sub = g3d[g3d['model'] == m]
    print(f'{MODEL_LABELS[m]:<14} {sub["gravy"].mean():>8.3f} '
          f'{sub["charge_at_pH7"].mean():>8.2f} '
          f'{sub["tempro_tm"].mean():>8.2f}')

plt.tight_layout()
plt.savefig('figures/fig4_3d_landscape.png', dpi=300, bbox_inches='tight')
plt.show()
print('\nSaved: figures/fig4_3d_landscape.png')


In [ ]:
# ── Figure 4c — Per-target Tm distribution: data-driven target selection ──
#
# Rationale for 4 targets (from all ~61):
#   T_dom:   Target where GDPO(DPO) most dominates all other models (clear winner)
#   T_worst: Target where GDPO(DPO) achieves lowest mean Tm (hardest case)
#   T_gain:  Target with largest GDPO(DPO) - SFT Tm gain (biggest improvement)
#   T_sft:   Target where GDPO(SFT) beats GDPO(DPO) most (alternative path wins)
#
# Data: seed=42, T=0.7 — consistent with Figure 4a/4b.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

_t, _seed = 0.7, 42
aa_models = ['SFT', 'DPO', 'GDPO', 'GDPO_SFT']

def is_valid_seq(seq):
    if not isinstance(seq, str): return False
    return len(seq) == 126 and seq.endswith('GGGGS') and seq.count('C') == 2

pool = valid_all[
    (valid_all['seed'] == _seed) & (valid_all['temperature'] == _t)
].copy()
pool = pool[pool['generated_sequence'].apply(is_valid_seq)].copy()
print(f'Data: seed={_seed}, T={_t}, {len(pool)} valid sequences')

target_means = (
    pool.groupby(['peptide', 'model'])['tempro_tm']
    .mean()
    .unstack('model')
    .dropna()
)

# GDPO dominance = how much GDPO beats the NEXT best model (max of SFT, DPO, GDPO_SFT)
target_means['next_best'] = target_means[['SFT','DPO','GDPO_SFT']].max(axis=1)
target_means['gdpo_dominance'] = target_means['GDPO'] - target_means['next_best']
target_means['gain'] = target_means['GDPO'] - target_means['SFT']
target_means['sft_path_adv'] = target_means['GDPO_SFT'] - target_means['GDPO']

# Select 4 distinct targets
selected = set()

t_dom = target_means['gdpo_dominance'].idxmax()
selected.add(t_dom)

t_worst = target_means['GDPO'].idxmin()
if t_worst in selected:
    t_worst = target_means.drop(index=selected)['GDPO'].idxmin()
selected.add(t_worst)

t_gain = target_means.drop(index=selected)['gain'].idxmax()
selected.add(t_gain)

t_sft = target_means.drop(index=selected)['sft_path_adv'].idxmax()
selected.add(t_sft)

FOCUS_TARGETS = [
    (t_dom,   'GDPO dominates\n(vs all others)'),
    (t_worst, 'Hardest target\n(lowest GDPO Tm)'),
    (t_gain,  'Largest gain\n(GDPO\u2212SFT)'),
    (t_sft,   'SFT-path wins\n(GDPO_SFT>GDPO)'),
]

print('Selected 4 targets from', len(target_means), 'total:')
print(f"{'Label':<24} {'SFT':>6} {'DPO':>6} {'GDPO':>6} {'G_SFT':>6} {'Dom':>6} {'Gain':>6}")
print('-' * 80)
for pep, lbl in FOCUS_TARGETS:
    r = target_means.loc[pep]
    ls = lbl.replace(chr(10), ' ')
    print(f'{ls:<24} {r["SFT"]:>6.1f} {r["DPO"]:>6.1f} {r["GDPO"]:>6.1f} '
          f'{r["GDPO_SFT"]:>6.1f} {r["gdpo_dominance"]:>+6.1f} {r["gain"]:>+6.1f}')
    print(f'  peptide: {pep}')

n_models  = len(aa_models)
n_targets = len(FOCUS_TARGETS)

all_tms = {}
for pep, tlbl in FOCUS_TARGETS:
    for m in aa_models:
        vals = pool[
            (pool['peptide'] == pep) & (pool['model'] == m)
        ]['tempro_tm'].dropna().values
        all_tms[(pep, m)] = vals

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 7))

box_width  = 0.16
group_gap  = 1.2
positions  = []
colors_all = []
data_all   = []
label_pos  = []
labels_x   = []

for ti, (pep, tlbl) in enumerate(FOCUS_TARGETS):
    center = ti * group_gap
    print(f'Target-{ti + 1}, {pep}')
    offsets = np.linspace(-(n_models-1)/2, (n_models-1)/2, n_models) * (box_width + 0.04)
    for mi, m in enumerate(aa_models):
        pos = center + offsets[mi]
        positions.append(pos)
        colors_all.append(MODEL_COLORS[m])
        data_all.append(all_tms[(pep, m)])
    label_pos.append(center)
    labels_x.append(f'Target-{ti + 1}')

bp = ax.boxplot(data_all, positions=positions, widths=box_width,
                patch_artist=True, notch=False, showfliers=False,
                medianprops=dict(color='white', linewidth=2.2),
                whiskerprops=dict(linewidth=1.0, color='#555'),
                capprops=dict(linewidth=1.0, color='#555'))

for i, (patch, fc) in enumerate(zip(bp['boxes'], colors_all)):
    patch.set_facecolor(fc)
    patch.set_alpha(0.80)
    patch.set_edgecolor('white')
    patch.set_linewidth(0.6)

rng = np.random.default_rng(42)
for i, (pos, vals, fc) in enumerate(zip(positions, data_all, colors_all)):
    jitter = rng.uniform(-box_width*0.35, box_width*0.35, len(vals))
    ax.scatter(pos + jitter, vals, s=2, color=fc, alpha=0.12,
               zorder=0, linewidths=0)

for ti in range(n_targets - 1):
    mid = (ti * group_gap + (ti+1) * group_gap) / 2
    ax.axvline(mid, color='#ddd', linewidth=0.8, linestyle='-', zorder=0)

ax.set_xticks(label_pos)
ax.set_xticklabels(labels_x, fontsize=13)
ax.tick_params(axis='y', labelsize=12)
ax.set_ylabel('Tm  (\u00b0C)', fontsize=14)
ax.grid(axis='y', alpha=0.20, linestyle='--')

legend_patches = [mpatches.Patch(color=MODEL_COLORS[m], label=MODEL_LABELS[m], alpha=0.8)
                  for m in aa_models]
ax.legend(handles=legend_patches, fontsize=9, framealpha=0.92,
          loc='upper left', ncol=2, handlelength=1.5)

plt.tight_layout()
plt.savefig('figures/fig4c_target_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()
print('\nSaved: figures/fig4c_target_boxplots.png')


In [ ]:
# ── Figure 4d — Per-target Solubility distribution (same 4 targets as 4c) ──
# Data: seed=42, T=0.7 — consistent with Figure 4a/4b/4c.
# Reuses pool and FOCUS_TARGETS from the cell above.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

PROP_COL  = 'netsolp_solubility'
PROP_NAME = 'NetSolP Solubility'

n_models  = len(aa_models)
n_targets = len(FOCUS_TARGETS)

all_vals = {}
for pep, tlbl in FOCUS_TARGETS:
    for m in aa_models:
        vals = pool[
            (pool['peptide'] == pep) & (pool['model'] == m)
        ][PROP_COL].dropna().values
        all_vals[(pep, m)] = vals

fig, ax = plt.subplots(figsize=(15, 7))

box_width  = 0.16
group_gap  = 1.2
positions  = []
colors_all = []
data_all   = []
label_pos  = []
labels_x   = []

for ti, (pep, tlbl) in enumerate(FOCUS_TARGETS):
    center = ti * group_gap
    offsets = np.linspace(-(n_models-1)/2, (n_models-1)/2, n_models) * (box_width + 0.04)
    for mi, m in enumerate(aa_models):
        pos = center + offsets[mi]
        positions.append(pos)
        colors_all.append(MODEL_COLORS[m])
        data_all.append(all_vals[(pep, m)])
    label_pos.append(center)
    labels_x.append(f'Target-{ti + 1}')

bp = ax.boxplot(data_all, positions=positions, widths=box_width,
                patch_artist=True, notch=False, showfliers=False,
                medianprops=dict(color='white', linewidth=2.2),
                whiskerprops=dict(linewidth=1.0, color='#555'),
                capprops=dict(linewidth=1.0, color='#555'))

for i, (patch, fc) in enumerate(zip(bp['boxes'], colors_all)):
    patch.set_facecolor(fc)
    patch.set_alpha(0.80)
    patch.set_edgecolor('white')
    patch.set_linewidth(0.6)

rng = np.random.default_rng(42)
for i, (pos, vals, fc) in enumerate(zip(positions, data_all, colors_all)):
    jitter = rng.uniform(-box_width*0.35, box_width*0.35, len(vals))
    ax.scatter(pos + jitter, vals, s=2, color=fc, alpha=0.12,
               zorder=0, linewidths=0)

for ti in range(n_targets - 1):
    mid = (ti * group_gap + (ti+1) * group_gap) / 2
    ax.axvline(mid, color='#ddd', linewidth=0.8, linestyle='-', zorder=0)

ax.set_xticks(label_pos)
ax.set_xticklabels(labels_x, fontsize=13)
ax.tick_params(axis='y', labelsize=12)
ax.set_ylabel(f'{PROP_NAME}', fontsize=14)
ax.grid(axis='y', alpha=0.20, linestyle='--')

legend_patches = [mpatches.Patch(color=MODEL_COLORS[m], label=MODEL_LABELS[m], alpha=0.8)
                  for m in aa_models]
ax.legend(handles=legend_patches, fontsize=9, framealpha=0.92,
          loc='upper left', ncol=2, handlelength=1.5)

plt.tight_layout()
plt.savefig('figures/fig4d_solubility_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()
print('\nSaved: figures/fig4d_solubility_boxplots.png')

In [ ]:
# ── GRAVY vs Solubility: path-dependent hydrophobicity explains variance ──
# Shows that GDPO(DPO)'s hydrophobic shift widens solubility spread,
# while GDPO(SFT) stays in a tighter, more hydrophilic zone.
# Uses seed=42, T=0.7 — consistent with Figure 4a/4b.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse

_t, _seed = 0.7, 42
aa_models = ['SFT', 'DPO', 'GDPO', 'GDPO_SFT']

def is_valid_seq(seq):
    if not isinstance(seq, str): return False
    return len(seq) == 126 and seq.endswith('GGGGS') and seq.count('C') == 2

pool = valid_all[
    (valid_all['seed'] == _seed) & (valid_all['temperature'] == _t)
].copy()
pool = pool[pool['generated_sequence'].apply(is_valid_seq)].copy()
print(f'Data: seed={_seed}, T={_t}, {len(pool)} valid sequences')

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ── Panel A: Per-sequence GRAVY vs Solubility scatter (per-target means) ──
ax = axes[0]
targets = pool['peptide'].unique()

for m in aa_models:
    gravys, sols = [], []
    for pep in targets:
        sub = pool[(pool['peptide'] == pep) & (pool['model'] == m)]
        if len(sub) < 10:
            continue
        gravys.append(sub['gravy'].mean())
        sols.append(sub['netsolp_solubility'].mean())
    ax.scatter(gravys, sols, color=MODEL_COLORS[m], s=30, alpha=0.55,
               label=MODEL_LABELS[m], edgecolors='white', linewidths=0.3)

    if len(gravys) > 2:
        gx, gy = np.array(gravys), np.array(sols)
        cx, cy = gx.mean(), gy.mean()
        cov = np.cov(gx, gy)
        eigvals, eigvecs = np.linalg.eigh(cov)
        order = eigvals.argsort()[::-1]
        eigvals, eigvecs = eigvals[order], eigvecs[:, order]
        angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
        w, h = 2 * 1.5 * np.sqrt(eigvals)
        ell = Ellipse((cx, cy), w, h, angle=angle,
                       facecolor=MODEL_COLORS[m], alpha=0.12, edgecolor=MODEL_COLORS[m],
                       linewidth=1.5, linestyle='--')
        ax.add_patch(ell)
        ax.plot(cx, cy, 'X', color=MODEL_COLORS[m], markersize=10,
                markeredgecolor='white', markeredgewidth=1.2, zorder=5)

ax.set_xlabel('Mean GRAVY per target\n← hydrophilic | hydrophobic →', fontsize=10)
ax.set_ylabel('Mean NetSolP Solubility per target', fontsize=10)
ax.set_title('(A)  Hydrophobicity vs Solubility\nper-target means across all ~61 targets',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8, loc='lower left', framealpha=0.9)
ax.grid(alpha=0.15, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Panel B: Per-target solubility std vs per-target mean GRAVY ──
ax2 = axes[1]
for m in aa_models:
    gravys, sol_stds = [], []
    for pep in targets:
        sub = pool[(pool['peptide'] == pep) & (pool['model'] == m)]
        if len(sub) < 20:
            continue
        gravys.append(sub['gravy'].mean())
        sol_stds.append(sub['netsolp_solubility'].std())
    ax2.scatter(gravys, sol_stds, color=MODEL_COLORS[m], s=30, alpha=0.55,
                label=MODEL_LABELS[m], edgecolors='white', linewidths=0.3)

    if len(gravys) > 2:
        gx, gy = np.array(gravys), np.array(sol_stds)
        cx, cy = gx.mean(), gy.mean()
        ax2.plot(cx, cy, 'X', color=MODEL_COLORS[m], markersize=10,
                 markeredgecolor='white', markeredgewidth=1.2, zorder=5)

ax2.set_xlabel('Mean GRAVY per target\n← hydrophilic | hydrophobic →', fontsize=10)
ax2.set_ylabel('Solubility std per target', fontsize=10)
ax2.set_title('(B)  Hydrophobicity vs Solubility Variance\nmore hydrophobic → wider solubility spread?',
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=8, loc='upper left', framealpha=0.9)
ax2.grid(alpha=0.15, linestyle='--')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# ── Panel C: Model-level summary bar chart ──
ax3 = axes[2]
bar_width = 0.18
x_pos = np.arange(len(aa_models))

mean_gravys = []
mean_sol_stds = []
for m in aa_models:
    sub = pool[pool['model'] == m]
    per_target = sub.groupby('peptide').agg(
        g=('gravy', 'mean'), s=('netsolp_solubility', 'std')
    ).dropna()
    mean_gravys.append(per_target['g'].mean())
    mean_sol_stds.append(per_target['s'].mean())

bar_colors = [MODEL_COLORS[m] for m in aa_models]
bar_labels = [MODEL_LABELS[m] for m in aa_models]

ax3_twin = ax3.twinx()

bars1 = ax3.bar(x_pos - bar_width/2, mean_gravys, bar_width * 0.9,
                color=bar_colors, alpha=0.7, edgecolor='white', linewidth=0.8)
bars2 = ax3_twin.bar(x_pos + bar_width/2, mean_sol_stds, bar_width * 0.9,
                     color=bar_colors, alpha=0.35, edgecolor=bar_colors,
                     linewidth=1.5, linestyle='--', hatch='///')

ax3.set_xticks(x_pos)
ax3.set_xticklabels(bar_labels, fontsize=9)
ax3.set_ylabel('Mean GRAVY (solid bars)', fontsize=9, color='#333')
ax3_twin.set_ylabel('Mean sol. std (hatched bars)', fontsize=9, color='#666')
ax3.set_title('(C)  Model-level summary\nGRAVY ↔ Solubility variance',
              fontsize=11, fontweight='bold')
ax3.grid(axis='y', alpha=0.15, linestyle='--')
ax3.spines['top'].set_visible(False)
ax3_twin.spines['top'].set_visible(False)

plt.tight_layout()
plt.savefig('figures/gravy_vs_solubility_path_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Summary statistics ──
print('\n── GRAVY & Solubility Variance Summary (per-target means) ──')
print(f"{'Model':<16} {'Mean GRAVY':>12} {'Mean Sol. Std':>14}")
print('-' * 44)
for m, g, s in zip(aa_models, mean_gravys, mean_sol_stds):
    print(f'{MODEL_LABELS[m]:<16} {g:>12.4f} {s:>14.4f}')

print(f'\nΔ GRAVY:  GDPO(DPO) − GDPO(SFT) = {mean_gravys[2] - mean_gravys[3]:+.4f}')
print(f'Δ Sol.σ:  GDPO(DPO) − GDPO(SFT) = {mean_sol_stds[2] - mean_sol_stds[3]:+.4f}')
print(f'\nInterpretation: GDPO(DPO) is more hydrophobic (higher GRAVY)')
print(f'and shows wider solubility variance than GDPO(SFT).')
print(f'The DPO step shifts CDR composition toward hydrophobic residues;')
print(f'GDPO(SFT) bypasses this, staying in a tighter solubility-safe zone.')
print(f'\nSaved: figures/gravy_vs_solubility_path_analysis.png')

In [ ]:
# ── Solubility vs Liability vs GRAVY: 3-way property landscape ──────────────
# Shows how the rewarded property (liability) and unrewarded properties
# (solubility, GRAVY) co-vary across models and targets.
# Uses seed=42, T=0.7 — consistent with Figure 4a/4b.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

_t, _seed = 0.7, 42
aa_models = ['SFT', 'DPO', 'GDPO', 'GDPO_SFT']

def is_valid_seq(seq):
    if not isinstance(seq, str): return False
    return len(seq) == 126 and seq.endswith('GGGGS') and seq.count('C') == 2

pool = valid_all[
    (valid_all['seed'] == _seed) & (valid_all['temperature'] == _t)
].copy()
pool = pool[pool['generated_sequence'].apply(is_valid_seq)].copy()
targets = pool['peptide'].unique()
print(f'Data: seed={_seed}, T={_t}, {len(pool)} valid sequences')

def draw_ellipse(ax, x_arr, y_arr, color, sigma=1.5):
    cx, cy = x_arr.mean(), y_arr.mean()
    cov = np.cov(x_arr, y_arr)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    w, h = 2 * sigma * np.sqrt(eigvals)
    ell = Ellipse((cx, cy), w, h, angle=angle,
                   facecolor=color, alpha=0.12, edgecolor=color,
                   linewidth=1.5, linestyle='--')
    ax.add_patch(ell)
    ax.plot(cx, cy, 'X', color=color, markersize=10,
            markeredgecolor='white', markeredgewidth=1.2, zorder=5)

fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))

# ── Panel A: Liability (x) vs Solubility (y) ──
ax = axes[0]
for m in aa_models:
    liabs, sols = [], []
    for pep in targets:
        sub = pool[(pool['peptide'] == pep) & (pool['model'] == m)]
        if len(sub) < 10:
            continue
        liabs.append(sub['liability_severity'].mean())
        sols.append(sub['netsolp_solubility'].mean())
    x_arr, y_arr = np.array(liabs), np.array(sols)
    ax.scatter(x_arr, y_arr, color=MODEL_COLORS[m], s=30, alpha=0.55,
               label=MODEL_LABELS[m], edgecolors='white', linewidths=0.3)
    if len(x_arr) > 2:
        draw_ellipse(ax, x_arr, y_arr, MODEL_COLORS[m])

ax.set_xlabel('Mean Liability Severity per target\n← higher | lower (better) →', fontsize=10)
ax.set_ylabel('Mean NetSolP Solubility per target', fontsize=10)
ax.invert_xaxis()
ax.legend(fontsize=10, loc='best', framealpha=0.9)
ax.grid(alpha=0.15, linestyle='--')
for spine in ax.spines.values():
    spine.set_visible(True)

# ── Panel B: Liability (x) vs GRAVY (y) ──
ax2 = axes[1]
for m in aa_models:
    liabs, gravys = [], []
    for pep in targets:
        sub = pool[(pool['peptide'] == pep) & (pool['model'] == m)]
        if len(sub) < 10:
            continue
        liabs.append(sub['liability_severity'].mean())
        gravys.append(sub['gravy'].mean())
    x_arr, y_arr = np.array(liabs), np.array(gravys)
    ax2.scatter(x_arr, y_arr, color=MODEL_COLORS[m], s=30, alpha=0.55,
                label=MODEL_LABELS[m], edgecolors='white', linewidths=0.3)
    if len(x_arr) > 2:
        draw_ellipse(ax2, x_arr, y_arr, MODEL_COLORS[m])

ax2.set_xlabel('Mean Liability Severity per target\n← higher | lower (better) →', fontsize=10)
ax2.set_ylabel('Mean GRAVY per target\n← hydrophilic | hydrophobic →', fontsize=10)
ax2.invert_xaxis()
ax2.legend(fontsize=10, loc='best', framealpha=0.9)
ax2.grid(alpha=0.15, linestyle='--')
for spine in ax2.spines.values():
    spine.set_visible(True)

# ── Panel C: GRAVY (x) vs Solubility (y) — colored by liability ──
ax3 = axes[2]
for m in aa_models:
    gravys, sols, liabs = [], [], []
    for pep in targets:
        sub = pool[(pool['peptide'] == pep) & (pool['model'] == m)]
        if len(sub) < 10:
            continue
        gravys.append(sub['gravy'].mean())
        sols.append(sub['netsolp_solubility'].mean())
        liabs.append(sub['liability_severity'].mean())
    x_arr, y_arr = np.array(gravys), np.array(sols)
    ax3.scatter(x_arr, y_arr, color=MODEL_COLORS[m], s=30, alpha=0.55,
                label=MODEL_LABELS[m], edgecolors='white', linewidths=0.3)
    if len(x_arr) > 2:
        draw_ellipse(ax3, x_arr, y_arr, MODEL_COLORS[m])

ax3.set_xlabel('Mean GRAVY per target\n← hydrophilic | hydrophobic →', fontsize=10)
ax3.set_ylabel('Mean NetSolP Solubility per target', fontsize=10)
ax3.legend(fontsize=10, loc='best', framealpha=0.9)
ax3.grid(alpha=0.15, linestyle='--')
for spine in ax3.spines.values():
    spine.set_visible(True)

plt.tight_layout()
plt.savefig('figures/liability_solubility_gravy_landscape.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Summary ──
print('\n── Model Centroids: Liability × Solubility × GRAVY ──')
print(f"{'Model':<16} {'Liability':>10} {'Solubility':>11} {'GRAVY':>8}")
print('-' * 48)
for m in aa_models:
    sub = pool[pool['model'] == m]
    per_t = sub.groupby('peptide').agg(
        l=('liability_severity', 'mean'),
        s=('netsolp_solubility', 'mean'),
        g=('gravy', 'mean')
    ).dropna()
    print(f'{MODEL_LABELS[m]:<16} {per_t["l"].mean():>10.2f} {per_t["s"].mean():>11.4f} {per_t["g"].mean():>8.4f}')

print('\n── Interpretation ──')
print('Panel A: Both GDPO models reduced liability (left shift),')
print('         but GDPO(SFT) also jumped UP in solubility.')
print('         GDPO(DPO) reduced liability without solubility gain.')
print('Panel B: Liability reduction correlated with hydrophobic shift')
print('         for GDPO(DPO) but not for GDPO(SFT).')
print('Panel C: Confirms GDPO(SFT) found a unique optimum —')
print('         moderate hydrophobicity + high solubility.')
print('\nSaved: figures/liability_solubility_gravy_landscape.png')